# PTD Scraper - Planos de Transformação Digital (Gov.br)

Notebook para coleta, extração e análise dos Planos de Transformação Digital dos órgãos federais brasileiros.

**Fonte:** [gov.br/governodigital - Planos de Transformação Digital](https://www.gov.br/governodigital/pt-br/estrategias-e-governanca-digital/planos-de-transformacao-digital)

**Pipeline:**
1. Scraping da lista de órgãos signatários e URLs dos PDFs
2. Download dos PDFs (Documento Diretivo + Anexo de Entregas)
3. Extração de tabelas (PyMuPDF `find_tables()` / Docling para OCR)
4. Padronização de vocabulário
5. Exportação CSV/JSON + relatório de erros
6. Análises estatísticas do corpus

In [ ]:
# ============================================================
# CÉLULA 1 — Setup & Instalação
# ============================================================
# Detecta ambiente (Google Colab ou local) e instala dependências.
# Execute esta célula primeiro.

import os, sys

# --- Detecção de ambiente ---
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Instalação de pacotes (apenas no Colab; local usa requirements.txt) ---
if IN_COLAB:
    get_ipython().system('pip install -q docling beautifulsoup4 requests tqdm pandas matplotlib seaborn')

# --- Google Drive (persistência de PDFs e outputs no Colab) ---
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/PTD_Scraper"
else:
    BASE_DIR = os.path.join(os.getcwd(), "ptd_output")

DIRS = {
    "pdfs_diretivo":  os.path.join(BASE_DIR, "pdfs", "diretivo"),
    "pdfs_entregas":  os.path.join(BASE_DIR, "pdfs", "entregas"),
    "output":         os.path.join(BASE_DIR, "output"),
    "checkpoints":    os.path.join(BASE_DIR, "checkpoints"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print(f"Ambiente: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Diretório base: {BASE_DIR}")
print("Estrutura criada:", list(DIRS.keys()))

In [ ]:
# ============================================================
# CÉLULA 2 — Configuração, Constantes e Estruturas de Dados
# ============================================================
import os, sys, time, pickle, unicodedata, re, json, logging, difflib
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Tuple, Dict, Any
from pathlib import Path
from datetime import datetime

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# --------------- URLs e parâmetros de rede ------------------
BASE_URL = ("https://www.gov.br/governodigital/pt-br/"
            "estrategias-e-governanca-digital/"
            "planos-de-transformacao-digital")
REQUEST_DELAY = 2.0        # segundos entre requests ao gov.br
MAX_RETRIES   = 4
REQUEST_TIMEOUT = 90
HTTP_HEADERS  = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0 Safari/537.36"),
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.5",
}

# --------------- Vocabulários canônicos ---------------------
CANONICAL_EIXOS = [
    "Serviços Digitais e Melhoria da Qualidade",
    "Unificação de Canais Digitais",
    "Governança e Gestão de Dados",
    "Segurança e Privacidade",
    "Projetos Especiais",
]

# Mapa Eixo → lista de Produtos (44 produtos oficiais)
CANONICAL_PRODUTOS: Dict[str, List[str]] = {
    "Serviços Digitais e Melhoria da Qualidade": [
        "Disponibilização em Acesso Digital",
        "Disponibilização de datas/cronograma na Agenda Gov.Br",
        "Evolução do Serviço",
        "Implantação da Área Logada Gov.Br",
        "Implantação da Experiência LabQ",
        "Implementação do VLIBRAS",
        "Integração à ferramenta de acompanhamento das solicitações",
        "Integração à ferramenta de avaliação da satisfação dos usuários",
        "Realização de Autodiagnóstico de Qualidade",
        "Revisão da descrição dos serviços",
        "Oferta de agendamento digital",
        "Integração à ferramenta de solicitação de atendimento presencial",
        "Implantação do Atendimento Virtual",
    ],
    "Unificação de Canais Digitais": [
        "Implantação do Design System",
        "Integração ao Login Único",
        "Integração ao Pagtesouro",
        "Integração com Assinatura Digital Gov.br",
        "Migração de APPs móveis para a loja do Gov.br",
        "Migração de Portal Institucional",
        "Migração de Serviço para Plataforma Unificada",
    ],
    "Governança e Gestão de Dados": [
        "Abertura de bases de dados",
        "Adequação à LGPD",
        "Catalogação de bases de dados no Portal de Dados",
        "Criação de Comitê de Governança de Dados",
        "Elaboração de Plano de Dados Abertos",
        "Elaboração do PDTIC",
        "Elaboração/Revisão da POSIC",
        "Implantação de processo de gestão de dados",
        "Implantação do inventário de bases de dados",
        "Implementação da interoperabilidade de dados",
        "Integração de dados ao Conecta Gov.br",
        "Integração de dados ao Datamart",
        "Melhoria da qualidade de bases de dados",
        "Nomeação de Encarregado de Dados Pessoais",
        "Obtenção de certificação de bases de dados",
        "Publicação de dados no Portal de Dados Abertos",
        "Realização de Inventário de Dados",
        "Relatório de Impacto à Proteção de Dados Pessoais",
        "Resposta a demandas de compartilhamento",
        "Utilização de dados do Conecta Gov.br",
    ],
    "Segurança e Privacidade": [
        "Adequação ao Framework de Privacidade e Segurança da Informação",
        "Elevação do nível de maturidade em privacidade e segurança",
    ],
    "Projetos Especiais": [
        "Iniciativa de Transformação Digital",
        "Projeto Especial de Transformação Digital",
    ],
}

# ---------- Produtos de templates anteriores (v1.x, v2.x) -----
# Produtos que aparecem em PTDs mais antigos e não estão no template v4.0
# Mapeados ao canônico mais próximo ou mantidos como produto válido extra
LEGACY_PRODUTOS: Dict[str, List[str]] = {
    "Segurança e Privacidade": [
        "Implementação do PPSI",                             # template v2.x
        "Adequação ao PPSI",                                 # variante
        "Auto-avaliação, análise de lacunas e planejamento do PPSI",
    ],
    "Governança e Gestão de Dados": [
        "Integração à base de dados",                        # template v2.x genérico
        "Interoperabilidade de Sistemas",                    # eixo antigo EGD 2020
        "Compartilhamento de dados via Conecta Gov.br",      # variante
    ],
    "Serviços Digitais e Melhoria da Qualidade": [
        "Digitalização de Serviço",                          # EGD 2020-2022
        "Publicação de Serviço no Portal Gov.br",            # EGD 2020-2022
        "Transformação Digital de Serviço",                   # EGD 2020-2022
    ],
    "Projetos Especiais": [
        "Ação estratégica de transformação digital",         # variante
        "Outros",                                            # produto genérico usado por 39 órgãos
    ],
}

# ---------- Aliases: texto variante → canônico exato ----------
# Mapeamento determinístico de variações conhecidas
PRODUTO_ALIASES: Dict[str, str] = {
    # Truncamentos e variações de acentuação
    "Integração à ferramenta de avaliação da satisfação dos usuários dos serviços":
        "Integração à ferramenta de avaliação da satisfação dos usuários",
    "Evolução do Serviço Digital":
        "Evolução do Serviço",
    "Integração ao Login Único Gov.Br":
        "Integração ao Login Único",
    "Integração ao Login Unico":
        "Integração ao Login Único",
    "Implementação do VLibras":
        "Implementação do VLIBRAS",
    "Implementacao do VLIBRAS":
        "Implementação do VLIBRAS",
    "Implantação da Area Logada Gov.Br":
        "Implantação da Área Logada Gov.Br",
    "Migração de Serviço para Plataforma Unificada Gov.br":
        "Migração de Serviço para Plataforma Unificada",
    "Migração do Portal Institucional":
        "Migração de Portal Institucional",
    "Adequação à Lei Geral de Proteção de Dados":
        "Adequação à LGPD",
    "Elaboração do Plano Diretor de TIC":
        "Elaboração do PDTIC",
    "Elaboração da POSIC":
        "Elaboração/Revisão da POSIC",
    "Revisão da POSIC":
        "Elaboração/Revisão da POSIC",
    "Integração à ferramenta de acompanhamento de solicitações":
        "Integração à ferramenta de acompanhamento das solicitações",
    "Disponibilização em acesso digital":
        "Disponibilização em Acesso Digital",
    "Revisão da descrição de serviços":
        "Revisão da descrição dos serviços",
    # Legacy / EGD 2020 mappings
    "Digitalização de Serviço":
        "Disponibilização em Acesso Digital",
    "Publicação de Serviço no Portal Gov.br":
        "Disponibilização em Acesso Digital",
    "Transformação Digital de Serviço":
        "Disponibilização em Acesso Digital",
}

# ---------- Eixos legados (EGD 2020-2022) → canônico ----------
EIXO_ALIASES: Dict[str, str] = {
    "Transformação Digital de Serviços Públicos":
        "Serviços Digitais e Melhoria da Qualidade",
    "Transformação Digital dos Serviços":
        "Serviços Digitais e Melhoria da Qualidade",
    "Unificação de Canais Digitais e Plataformas":
        "Unificação de Canais Digitais",
    "Governo como Plataforma":
        "Unificação de Canais Digitais",
    "Governo Aberto e Transparência":
        "Governança e Gestão de Dados",
    "Infraestrutura de TIC e Governança de Dados":
        "Governança e Gestão de Dados",
    "Interoperabilidade de Sistemas":
        "Governança e Gestão de Dados",
    "Dados para o Desenvolvimento":
        "Governança e Gestão de Dados",
    "Identidade Digital do Cidadão":
        "Unificação de Canais Digitais",
    "Governo Inteligente":
        "Projetos Especiais",
}

# Lista flat de todos os produtos (canônicos + legados)
ALL_PRODUTOS = [p for prods in CANONICAL_PRODUTOS.values() for p in prods]
ALL_PRODUTOS += [p for prods in LEGACY_PRODUTOS.values() for p in prods]

# Mapa reverso: produto → eixo canônico (canônicos + legados)
PRODUTO_TO_EIXO = {}
for eixo, prods in CANONICAL_PRODUTOS.items():
    for p in prods:
        PRODUTO_TO_EIXO[p] = eixo
for eixo, prods in LEGACY_PRODUTOS.items():
    for p in prods:
        PRODUTO_TO_EIXO[p] = eixo

# Escalas do Documento Diretivo (tabela de riscos)
PROBABILIDADE_SCALE = [
    "raro", "pouco provável", "provável",
    "muito provável", "praticamente certo",
]
IMPACTO_SCALE = [
    "muito baixo", "baixo", "médio", "alto", "muito alto",
]
TRATAMENTO_OPTIONS = ["mitigar", "eliminar", "transferir", "aceitar"]

# ---------- Órgãos que compartilham PDFs (grupos) ----------
ORGAN_GROUPS: Dict[str, List[str]] = {
    "MD":   ["MD", "CEX", "CM", "COMAER", "CENSIPAM", "FOSORIO", "HFA"],
    "MEC":  ["MEC", "CAPES", "EBSERH", "FNDE", "FUNDAJ", "IBC", "INEP", "INES"],
    "MF":   ["MF", "RFB", "STN", "PGFN"],
    "MMA":  ["MMA", "IBAMA", "ICMBIO", "SFB", "JBRJ"],
    "MT":   ["MT", "ANTT", "DNIT"],
    "MIDR": ["MIDR", "CODEVASF", "SUDAM", "SUDECO", "SUDENE"],
    "MDA":  ["MDA", "CONAB"],
}
# Reverso: sigla membro → sigla cabeça do grupo
MEMBER_TO_GROUP = {}
for head, members in ORGAN_GROUPS.items():
    for m in members:
        if m != head:
            MEMBER_TO_GROUP[m] = head

# --------------- Dataclasses --------------------------------
@dataclass
class OrganInfo:
    sigla: str
    nome_completo: str
    grupo: Optional[str] = None          # sigla do cabeça, se agrupado
    url_diretivo: Optional[str] = None
    url_entregas: Optional[str] = None
    pdf_path_diretivo: Optional[str] = None
    pdf_path_entregas: Optional[str] = None

@dataclass
class RiskEntry:
    orgao_sigla: str
    risco_texto: str = ""
    probabilidade_original: str = ""
    probabilidade_normalizada: str = ""
    impacto_original: str = ""
    impacto_normalizado: str = ""
    tratamento_original: str = ""
    tratamento_normalizado: str = ""
    acoes_tratamento: str = ""
    extraction_confidence: str = "high"    # high / medium / low
    needs_review: bool = False
    review_reason: Optional[str] = None

@dataclass
class DeliveryEntry:
    orgao_sigla: str
    tabela_tipo: str = ""                  # pactuada / concluida / cancelada
    servico_acao: str = ""
    produto_original: str = ""
    produto_normalizado: str = ""
    eixo_original: str = ""
    eixo_normalizado: str = ""
    area_responsavel: Optional[str] = None
    data_pactuada: Optional[str] = None
    data_entrega: Optional[str] = None
    pactuado: Optional[str] = None         # Sim / Não (concluídas)
    justificativa: Optional[str] = None    # (canceladas)
    extraction_confidence: str = "high"
    needs_review: bool = False
    review_reason: Optional[str] = None

@dataclass
class ProcessingError:
    orgao_sigla: str
    document_type: str     # diretivo / entregas
    stage: str             # download / extraction / standardization
    error_type: str
    error_message: str
    timestamp: str = ""
    url: Optional[str] = None

    def __post_init__(self):
        if not self.timestamp:
            self.timestamp = datetime.now().isoformat()

# Contêineres globais
all_organs: List[OrganInfo] = []
all_risks: List[RiskEntry] = []
all_deliveries: List[DeliveryEntry] = []
all_errors: List[ProcessingError] = []

print(f"Configuração carregada: {len(ALL_PRODUTOS)} produtos canônicos em {len(CANONICAL_EIXOS)} eixos")

In [ ]:
# ============================================================
# CÉLULA 3 — Funções Utilitárias
# ============================================================

# --------------- Rede com retry -----------------------------
def safe_request(url: str, max_retries: int = MAX_RETRIES,
                 delay: float = REQUEST_DELAY,
                 timeout: int = REQUEST_TIMEOUT) -> Optional[requests.Response]:
    """GET com retry exponencial e rate-limiting."""
    for attempt in range(1, max_retries + 1):
        try:
            time.sleep(delay)
            resp = requests.get(url, headers=HTTP_HEADERS, timeout=timeout)
            if resp.status_code == 200:
                return resp
            if resp.status_code == 503 and attempt < max_retries:
                wait = delay * (2 ** attempt)
                print(f"  503 em {url} — retry {attempt}/{max_retries} em {wait:.0f}s")
                time.sleep(wait)
                continue
            resp.raise_for_status()
        except requests.RequestException as exc:
            if attempt < max_retries:
                wait = delay * (2 ** attempt)
                print(f"  Erro ({exc}) — retry {attempt}/{max_retries} em {wait:.0f}s")
                time.sleep(wait)
            else:
                print(f"  FALHA definitiva: {url} — {exc}")
                return None
    return None

# --------------- Normalização de texto ----------------------
def normalize_text(text: str) -> str:
    """Normaliza unicode, whitespace e caixa para comparação."""
    if not text:
        return ""
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def strip_accents(text: str) -> str:
    """Remove acentos para matching fuzzy."""
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

# --------------- Matching fuzzy de vocabulário --------------
def fuzzy_match(original: str, candidates: list,
                threshold: float = 0.85) -> Tuple[str, float]:
    """Retorna (melhor_candidato, score). Score 0 se abaixo do threshold."""
    if not original or not candidates:
        return ("", 0.0)
    norm = normalize_text(original).lower()
    norm_stripped = strip_accents(norm)
    # Tentativa exata (case-insensitive, accent-insensitive)
    for c in candidates:
        c_norm = normalize_text(c).lower()
        if norm == c_norm or norm_stripped == strip_accents(c_norm):
            return (c, 1.0)
    # Fuzzy
    best, best_score = "", 0.0
    for c in candidates:
        c_norm = normalize_text(c).lower()
        score = difflib.SequenceMatcher(None, norm_stripped,
                                        strip_accents(c_norm)).ratio()
        if score > best_score:
            best, best_score = c, score
    if best_score >= threshold:
        return (best, best_score)
    return (best, best_score)   # retorna melhor mesmo abaixo do threshold

def fuzzy_match_produto(original: str) -> Tuple[str, float]:
    """Match produto com: aliases determinísticos → canônicos+legados → fuzzy."""
    if not original:
        return ("", 0.0)
    norm = normalize_text(original)
    # Camada 0: alias determinístico (variações conhecidas)
    for alias_key, canonical_val in PRODUTO_ALIASES.items():
        if normalize_text(alias_key).lower() == norm.lower():
            return (canonical_val, 1.0)
        if strip_accents(normalize_text(alias_key).lower()) == strip_accents(norm.lower()):
            return (canonical_val, 0.98)
    # Camada 1+: match fuzzy contra lista completa (canônicos + legados)
    return fuzzy_match(original, ALL_PRODUTOS, threshold=0.80)

def fuzzy_match_eixo(original: str) -> Tuple[str, float]:
    """Match eixo com: aliases legados → canônicos → fuzzy."""
    if not original:
        return ("", 0.0)
    norm = normalize_text(original)
    # Camada 0: alias legado (eixos EGD 2020-2022 → EFGD 2024)
    for alias_key, canonical_val in EIXO_ALIASES.items():
        if normalize_text(alias_key).lower() == norm.lower():
            return (canonical_val, 0.95)
        if strip_accents(normalize_text(alias_key).lower()) == strip_accents(norm.lower()):
            return (canonical_val, 0.93)
    return fuzzy_match(original, CANONICAL_EIXOS, threshold=0.80)

def fuzzy_match_scale(original: str, scale: list) -> Tuple[str, float]:
    return fuzzy_match(original, scale, threshold=0.70)

# --------------- Parse de datas variadas --------------------
_DATE_PATTERNS = [
    (r"(\d{2})/(\d{2})/(\d{4})", lambda m: f"{m.group(3)}-{m.group(2)}-{m.group(1)}"),
    (r"(\d{2})/(\d{4})",          lambda m: f"{m.group(2)}-{m.group(1)}"),
    (r"(\d{4})-(\d{2})-(\d{2})",  lambda m: m.group(0)),
]
_MONTH_MAP = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04",
    "mai": "05", "jun": "06", "jul": "07", "ago": "08",
    "set": "09", "out": "10", "nov": "11", "dez": "12",
}

def parse_date(text: str) -> Optional[str]:
    """Converte formatos variados para YYYY-MM ou YYYY-MM-DD."""
    if not text:
        return None
    text = normalize_text(text).strip()
    for pat, fmt in _DATE_PATTERNS:
        m = re.search(pat, text)
        if m:
            return fmt(m)
    # Formato "jun/25", "mar/2026"
    m = re.match(r"([a-záéíóú]{3})[\./\-](\d{2,4})", text.lower())
    if m:
        month = _MONTH_MAP.get(m.group(1)[:3])
        year = m.group(2)
        if len(year) == 2:
            year = "20" + year
        if month:
            return f"{year}-{month}"
    return text   # retorna original se não parsear

# --------------- Checkpoint / Resume ------------------------
def save_checkpoint(data: Any, name: str) -> None:
    path = os.path.join(DIRS["checkpoints"], f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint salvo: {name}")

def load_checkpoint(name: str) -> Optional[Any]:
    path = os.path.join(DIRS["checkpoints"], f"{name}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint carregado: {name}")
        return data
    return None

# --------------- Logging ------------------------------------
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ptd_scraper")

print("Funções utilitárias carregadas.")

## 2. Scraping — Lista de Órgãos Signatários

Extrai os 91 órgãos e seus links de PDFs (Documento Diretivo + Anexo de Entregas) da página principal.

In [ ]:
# ============================================================
# CÉLULA 4 — Scraping: Lista de Órgãos e URLs dos PDFs
# ============================================================

def _classify_pdf_link(anchor_text: str) -> Optional[str]:
    """Classifica um link de PDF como 'diretivo' ou 'entregas' pelo texto da âncora."""
    text = normalize_text(anchor_text).lower()
    text_no_accents = strip_accents(text)

    diretivo_kw = ["diretivo", "documento diretivo"]
    if any(kw in text_no_accents for kw in diretivo_kw):
        return "diretivo"

    entregas_kw = ["entregas", "anexo de entregas", "anexo entregas"]
    if any(kw in text_no_accents for kw in entregas_kw):
        return "entregas"

    return None


def _extract_siglas_from_header(header_text: str) -> List[str]:
    """
    Extrai siglas de um cabeçalho como:
      'Plano de Transformação Digital SIGLA:'
      'Plano de Transformação Digital SIGLA1 / SIGLA2 / SIGLA3:'
      'Plano de Transformação Digital CVM -'
    """
    text = normalize_text(header_text)

    # Remove o prefixo "Plano de Transformação Digital"
    prefix_pattern = re.compile(
        r"Plano\s+de\s+Transforma[çc][ãa]o\s+Digital\s*", re.IGNORECASE
    )
    text = prefix_pattern.sub("", text).strip()

    # Remove trailing : ou -
    text = re.sub(r"[\s:–\-]+$", "", text).strip()

    if not text:
        return []

    # Divide por " / " para múltiplas siglas
    parts = [p.strip() for p in re.split(r"\s*/\s*", text) if p.strip()]

    # Filtra: siglas são uppercase (permitem hífen para SG-PR), 2-14 chars
    siglas = []
    for p in parts:
        # Limpa possíveis sufixos ("(NOVO)")
        p = re.sub(r"\s*\(.*?\)\s*", "", p).strip()
        if re.match(r"^[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ0-9\-]{1,14}$", p):
            siglas.append(p)

    return siglas


def scrape_organ_listing(url: str) -> List[OrganInfo]:
    """
    Faz o scraping da página gov.br para extrair órgãos e links de PDFs.

    Estrutura real da página:
      Cada órgão é um <td> contendo:
        <strong>Plano de Transformação Digital SIGLA:</strong>
        <a href="...diretivo.pdf">Documento Diretivo</a> /
        <a href="...entregas.pdf">Anexo de Entregas</a>
    """
    resp = safe_request(url)
    if resp is None:
        raise RuntimeError(f"Não foi possível acessar {url}")

    soup = BeautifulSoup(resp.content, "html.parser")

    # Encontrar todos os <strong> que contêm "Plano de Transformação Digital"
    organ_data: Dict[str, Dict[str, Optional[str]]] = {}
    seen_sigla_sets = set()  # evitar duplicatas

    for strong_tag in soup.find_all(["strong", "b"]):
        raw_text = strong_tag.get_text(separator=" ", strip=True)
        if "transformação digital" not in raw_text.lower() and "transformacao digital" not in raw_text.lower():
            continue

        siglas = _extract_siglas_from_header(raw_text)
        if not siglas:
            continue

        # Evitar processar duplicatas do mesmo grupo
        sigla_key = tuple(sorted(siglas))
        if sigla_key in seen_sigla_sets:
            continue
        seen_sigla_sets.add(sigla_key)

        # Encontrar links PDF no mesmo container (td, p, div, etc.)
        container = strong_tag.parent
        if container is None:
            continue

        pdf_links_in_container = []
        for a_tag in container.find_all("a", href=True):
            href = a_tag["href"]
            if not href.lower().endswith(".pdf"):
                continue
            if "ptds-vigentes/" not in href and "planos-de-transformacao-digital" not in href:
                continue
            anchor_text = a_tag.get_text(separator=" ", strip=True)
            doc_type = _classify_pdf_link(anchor_text)

            # Fallback: classificar pelo nome do arquivo
            if doc_type is None:
                fname = href.rsplit("/", 1)[-1].lower()
                if "diretivo" in fname or "diretiv" in fname:
                    doc_type = "diretivo"
                elif "entregas" in fname or "anexo" in fname:
                    doc_type = "entregas"
                else:
                    doc_type = "unknown"

            # Converter URL relativa para absoluta
            if href.startswith("/"):
                href = "https://www.gov.br" + href

            pdf_links_in_container.append((href, doc_type))

        # Atribuir URLs por tipo
        url_diretivo = None
        url_entregas = None
        for href, doc_type in pdf_links_in_container:
            if doc_type == "diretivo" and url_diretivo is None:
                url_diretivo = href
            elif doc_type == "entregas" and url_entregas is None:
                url_entregas = href
            elif doc_type == "unknown":
                if url_diretivo is None:
                    url_diretivo = href
                elif url_entregas is None:
                    url_entregas = href

        # Registrar para todas as siglas neste header
        for sigla in siglas:
            if sigla not in organ_data:
                organ_data[sigla] = {
                    "nome": raw_text,
                    "url_diretivo": url_diretivo,
                    "url_entregas": url_entregas,
                }

    logger.info(f"Scraping direto: {len(organ_data)} siglas encontradas")

    # Expandir grupos: membros herdam PDFs do cabeça se não tiverem próprios
    expanded = dict(organ_data)
    for head_sigla, members in ORGAN_GROUPS.items():
        if head_sigla in organ_data:
            head_info = organ_data[head_sigla]
            for member in members:
                if member == head_sigla:
                    continue
                if member not in expanded:
                    expanded[member] = {
                        "nome": head_info["nome"],
                        "url_diretivo": head_info["url_diretivo"],
                        "url_entregas": head_info["url_entregas"],
                    }
                else:
                    if expanded[member]["url_diretivo"] is None:
                        expanded[member]["url_diretivo"] = head_info["url_diretivo"]
                    if expanded[member]["url_entregas"] is None:
                        expanded[member]["url_entregas"] = head_info["url_entregas"]

    # Construir lista final
    organs: List[OrganInfo] = []
    for sigla in sorted(expanded.keys()):
        info = expanded[sigla]
        grupo = MEMBER_TO_GROUP.get(sigla)
        organs.append(OrganInfo(
            sigla=sigla,
            nome_completo=info["nome"],
            grupo=grupo,
            url_diretivo=info.get("url_diretivo"),
            url_entregas=info.get("url_entregas"),
        ))

    return organs


# ---- Execução (sempre faz scraping fresco — leva ~3s) ----
print("Fazendo scraping da página principal...")
all_organs = scrape_organ_listing(BASE_URL)

# ---- Validação e Resumo ----
_n_total = len(all_organs)
_n_diretivo = sum(1 for o in all_organs if o.url_diretivo)
_n_entregas = sum(1 for o in all_organs if o.url_entregas)
_n_ambos = sum(1 for o in all_organs if o.url_diretivo and o.url_entregas)
_n_nenhum = sum(1 for o in all_organs if not o.url_diretivo and not o.url_entregas)
_n_grupos = sum(1 for o in all_organs if o.grupo is not None)

print(f"\n{'='*50}")
print(f"Total de órgãos encontrados: {_n_total}")
if _n_total < 80 or _n_total > 110:
    print(f"  ⚠ ATENÇÃO: esperados ~91 órgãos, encontrados {_n_total}")
else:
    print(f"  ✓ Contagem dentro do esperado (~91)")
print(f"  Com Documento Diretivo:    {_n_diretivo}")
print(f"  Com Anexo de Entregas:     {_n_entregas}")
print(f"  Com ambos:                 {_n_ambos}")
print(f"  Sem nenhum PDF:            {_n_nenhum}")
print(f"  Membros de grupo:          {_n_grupos}")
print(f"{'='*50}")

if _n_nenhum > 0:
    print("\nÓrgãos SEM nenhum PDF:")
    for o in all_organs:
        if not o.url_diretivo and not o.url_entregas:
            print(f"  - {o.sigla}")

print("\nAmostra (primeiros 10):")
for o in all_organs[:10]:
    print(f"  {o.sigla:12s} | dir={'Sim' if o.url_diretivo else '---'} "
          f"| ent={'Sim' if o.url_entregas else '---'} "
          f"| grupo={o.grupo or '—'}")

## 3. Download dos PDFs

Baixa todos os PDFs identificados com rate-limiting e verificação de integridade.

In [ ]:
# ============================================================
# CÉLULA 5 — Download dos PDFs
# ============================================================

PDF_MAGIC_BYTES = b"%PDF"
MIN_SUSPICIOUS_SIZE = 10 * 1024   # 10 KB
MIN_VALID_SIZE = 1000             # 1 KB (skip threshold para resume)


def _download_single_pdf(url: str, dest_path: str, sigla: str,
                         doc_type: str) -> Optional[ProcessingError]:
    """
    Baixa um único PDF. Retorna ProcessingError em caso de falha, None se OK.
    Pula o download se o arquivo já existe com tamanho > MIN_VALID_SIZE.
    """
    # Resume: pula se já existe e parece válido
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > MIN_VALID_SIZE:
        return None

    resp = safe_request(url)
    if resp is None:
        return ProcessingError(
            orgao_sigla=sigla,
            document_type=doc_type,
            stage="download",
            error_type="request_failed",
            error_message=f"Falha ao baixar após {MAX_RETRIES} tentativas",
            url=url,
        )

    content = resp.content

    # Verifica magic bytes
    if not content[:4].startswith(PDF_MAGIC_BYTES):
        return ProcessingError(
            orgao_sigla=sigla,
            document_type=doc_type,
            stage="download",
            error_type="invalid_pdf",
            error_message=f"Arquivo não começa com %PDF (magic bytes: {content[:8]!r})",
            url=url,
        )

    # Salva o arquivo
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    with open(dest_path, "wb") as f:
        f.write(content)

    # Alerta para arquivos suspeitamente pequenos
    file_size = len(content)
    if file_size < MIN_SUSPICIOUS_SIZE:
        logger.warning(f"  {sigla}/{doc_type}: arquivo pequeno ({file_size:,} bytes) — pode estar corrompido")

    return None


def download_all_pdfs(organs: List[OrganInfo]) -> List[ProcessingError]:
    """
    Baixa todos os PDFs (diretivo + entregas) dos órgãos.
    Atualiza organ.pdf_path_diretivo e organ.pdf_path_entregas.
    Retorna lista de erros.
    """
    errors: List[ProcessingError] = []
    downloaded_urls: Dict[str, str] = {}   # url → caminho local (evita re-download)

    # Contagem total de downloads potenciais
    total_downloads = sum(
        (1 if o.url_diretivo else 0) + (1 if o.url_entregas else 0)
        for o in organs
    )

    pbar = tqdm(total=total_downloads, desc="Baixando PDFs", unit="pdf")

    for organ in organs:
        for doc_type, url_attr, path_attr, target_dir in [
            ("diretivo", "url_diretivo", "pdf_path_diretivo", DIRS["pdfs_diretivo"]),
            ("entregas", "url_entregas", "pdf_path_entregas", DIRS["pdfs_entregas"]),
        ]:
            url = getattr(organ, url_attr)
            if url is None:
                continue

            filename = f"{organ.sigla}_{doc_type}.pdf"
            dest_path = os.path.join(target_dir, filename)

            # Se outro órgão do mesmo grupo já baixou este URL, reutiliza
            if url in downloaded_urls:
                existing_path = downloaded_urls[url]
                if os.path.exists(existing_path):
                    # Copia ou cria link — usamos cópia para robustez
                    if not os.path.exists(dest_path):
                        import shutil
                        shutil.copy2(existing_path, dest_path)
                    setattr(organ, path_attr, dest_path)
                    pbar.update(1)
                    continue

            err = _download_single_pdf(url, dest_path, organ.sigla, doc_type)
            if err is not None:
                errors.append(err)
            else:
                setattr(organ, path_attr, dest_path)
                downloaded_urls[url] = dest_path

            pbar.update(1)

    pbar.close()
    return errors


# ---- Execução ----
# Download já faz skip automático de PDFs existentes (resume embutido no _download_single_pdf)
if all_organs:
    print(f"Iniciando download de PDFs para {len(all_organs)} órgãos...")
    print("(PDFs já baixados serão reutilizados automaticamente)")
    download_errors = download_all_pdfs(all_organs)
    all_errors.extend(download_errors)
else:
    print("ERRO: Nenhum órgão encontrado. Verifique a célula de scraping.")
    download_errors = []

# ---- Resumo ----
_n_dir_ok = sum(1 for o in all_organs if o.pdf_path_diretivo and os.path.exists(o.pdf_path_diretivo))
_n_ent_ok = sum(1 for o in all_organs if o.pdf_path_entregas and os.path.exists(o.pdf_path_entregas))

_total_size = 0
_suspicious: List[str] = []
for o in all_organs:
    for p in [o.pdf_path_diretivo, o.pdf_path_entregas]:
        if p and os.path.exists(p):
            sz = os.path.getsize(p)
            _total_size += sz
            if sz < MIN_SUSPICIOUS_SIZE:
                _suspicious.append(f"{o.sigla}: {os.path.basename(p)} ({sz:,} bytes)")

print(f"\n{'='*50}")
print(f"Download concluído")
print(f"  Documento Diretivo OK:    {_n_dir_ok}")
print(f"  Anexo de Entregas OK:     {_n_ent_ok}")
print(f"  Erros de download:        {len(download_errors)}")
print(f"  Tamanho total:            {_total_size / (1024*1024):.1f} MB")
print(f"{'='*50}")

if _suspicious:
    print(f"\nArquivos suspeitamente pequenos (<{MIN_SUSPICIOUS_SIZE//1024} KB):")
    for s in _suspicious:
        print(f"  - {s}")

if download_errors:
    print(f"\nErros de download:")
    for e in download_errors:
        print(f"  - {e.orgao_sigla}/{e.document_type}: {e.error_type} — {e.error_message}")

## 4. Configuração do Docling e Classificação de Tabelas

Inicializa o pipeline de extração de tabelas Docling (IBM) e define funções de classificação.

In [ ]:
# ============================================================
# CÉLULA 6 — Configuração do Docling e Classificação de Tabelas
# ============================================================

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat

# --------------- Criação do conversor -------------------------

def create_converter(accurate: bool = True, ocr: bool = True) -> DocumentConverter:
    """Cria um DocumentConverter configurado para extração de tabelas.

    Args:
        accurate: Se True usa TableFormerMode.ACCURATE; senão FAST.
        ocr: Se True habilita OCR para PDFs escaneados.

    Returns:
        DocumentConverter pronto para uso.
    """
    pipeline_opts = PdfPipelineOptions()
    pipeline_opts.do_table_structure = True
    pipeline_opts.table_structure_options.mode = (
        TableFormerMode.ACCURATE if accurate else TableFormerMode.FAST
    )

    # OCR
    pipeline_opts.do_ocr = ocr

    # Desabilita features desnecessárias (compatível com diferentes versões do docling)
    for attr in ["generate_picture_images", "do_picture_description",
                 "do_picture_classification", "do_code_enrichment",
                 "do_formula_enrichment"]:
        if hasattr(pipeline_opts, attr):
            setattr(pipeline_opts, attr, False)

    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_opts),
    }

    return DocumentConverter(format_options=format_options)


# --------------- Classificação de tabelas diretivo ------------

def _normalize_header(text: str) -> str:
    """Normaliza texto de cabeçalho para comparação: minúscula, sem acentos, sem espaços extras."""
    if not isinstance(text, str):
        return ""
    t = normalize_text(text).lower()
    return strip_accents(t)


def classify_diretivo_table(df: pd.DataFrame) -> str:
    """Classifica uma tabela extraída de Documento Diretivo.

    Returns:
        'risk_table' — tabela de riscos e tratamento
        'organ_info' — informações institucionais
        'signature'  — bloco de assinaturas
        'unknown'    — não classificada
    """
    if df is None or df.empty:
        return "unknown"

    ncols = len(df.columns)
    # Normalizar headers — tratar quebras de linha e hífens de quebra
    def _clean_col(c):
        s = _normalize_header(str(c))
        s = s.replace("- ", "").replace("-\n", "")  # hífens de quebra: "DtPac-\ntuada"
        return s
    headers = [_clean_col(c) for c in df.columns]
    # Também verificar a primeira linha caso os headers sejam genéricos (0,1,2...)
    first_row_headers = []
    if len(df) > 0:
        first_row_headers = [_clean_col(v) for v in df.iloc[0]]

    all_headers = headers + first_row_headers

    combined = " ".join(all_headers)

    # --- Tabela de riscos: espera 3-8 colunas com keywords específicas ---
    risk_keywords = ["risco", "probabilidade", "impacto", "tratamento", "ocorrer"]
    risk_alt_keywords = ["evento", "classificacao", "severidade", "resposta", "acao", "acoes",
                         "id do risco", "descricao do risco", "opcao de tratamento"]
    risk_hits = sum(1 for kw in risk_keywords if kw in combined)
    risk_alt_hits = sum(1 for kw in risk_alt_keywords if kw in combined)

    # Match principal: 2+ keywords primárias
    if risk_hits >= 2 and 3 <= ncols <= 8:
        return "risk_table"
    # Match alternativo: 1 primária + 1 alternativa
    if risk_hits >= 1 and risk_alt_hits >= 1 and 3 <= ncols <= 8:
        return "risk_table"
    # Match relaxado: combinação tem "risco" + "tratamento" em qualquer posição
    if "risco" in combined and "tratamento" in combined and 3 <= ncols <= 8:
        return "risk_table"

    # --- Informações do órgão: keywords institucionais ---
    info_keywords = [
        "orgao", "ministerio", "secretaria", "sigla", "cnpj",
        "responsavel", "gestor", "dirigente", "titular", "autoridade",
        "instituicao", "vinculacao",
    ]
    info_hits = sum(1 for kw in info_keywords if kw in combined)
    if info_hits >= 2:
        return "organ_info"

    # --- Bloco de assinatura ---
    sig_keywords = ["assinatura", "assinado", "data", "nome", "cargo", "cpf"]
    sig_hits = sum(1 for kw in sig_keywords if kw in combined)
    if sig_hits >= 2 and ncols <= 4:
        return "signature"

    return "unknown"


# --------------- Classificação de tabelas entregas ------------

def classify_entregas_table(df: pd.DataFrame) -> str:
    """Classifica uma tabela extraída de Anexo de Entregas.

    Returns:
        'pactuadas'   — entregas pactuadas (agendadas)
        'concluidas'  — entregas concluídas
        'canceladas'  — entregas canceladas
        'unknown'     — não classificada
    """
    if df is None or df.empty:
        return "unknown"

    headers = [_clean_col(c) for c in df.columns]
    first_row_headers = []
    if len(df) > 0:
        first_row_headers = [_clean_col(v) for v in df.iloc[0]]

    all_headers = headers + first_row_headers
    combined = " ".join(all_headers)
    combined_compact = combined.replace(" ", "")

    has_area_resp = any(
        "area" in h and "responsavel" in h for h in all_headers
    ) or "area responsavel" in combined or "arearesponsavel" in combined.replace(" ", "")

    has_dt_pactuada = (
        "dtpactuada" in combined.replace(" ", "")
        or "data pactuada" in combined
        or "dt pactuada" in combined
        or "datapactuada" in combined.replace(" ", "")
    )

    has_dt_entrega = (
        "dtentrega" in combined.replace(" ", "")
        or "data entrega" in combined
        or "dt entrega" in combined
        or "dataentrega" in combined.replace(" ", "")
        or "data de entrega" in combined
    )

    has_pactuado_flag = (
        "pactuado?" in combined
        or "pactuado ?" in combined
        or ("pactuado" in combined and ("sim" in combined or "nao" in combined))
    )

    has_justificativa = (
        "justificativa" in combined
        or "motivo" in combined and "cancelamento" in combined
    )

    # Classificação por prioridade (canceladas são mais restritivas)
    # Canceladas: tem Justificativa
    if has_justificativa:
        return "canceladas"

    # Concluídas: tem DtEntrega ou coluna "Pactuado?"
    if has_dt_entrega or has_pactuado_flag:
        return "concluidas"

    # Pactuadas: tem Area Responsável E DtPactuada
    if has_area_resp and has_dt_pactuada:
        return "pactuadas"

    # Fallback: se tem DtPactuada mas sem Area (layout alternativo)
    if has_dt_pactuada:
        return "pactuadas"

    return "unknown"


# --------------- Detecção de formato legado ---------------------

def detect_legacy_format(result) -> Optional[str]:
    """Detecta se um PDF convertido usa formato legado (pré-EFGD 2024).

    Heurísticas:
      - Presença de seções numeradas como '1 – PUBLICAÇÃO DE SERVIÇOS'
      - Ausência de tabelas com headers padrão (Servico/Acao, Produto, Eixo)
      - Menção a 'EGD', 'SGD/ME', 'formalizado no SEI'

    Returns:
        'legacy_egd2020' se formato antigo detectado, None se formato atual.
    """
    try:
        doc_text = result.document.export_to_markdown()[:5000].lower()
    except Exception:
        return None

    legacy_signals = [
        "publicação de serviços no portal gov.br",
        "transformação digital dos serviços no portal gov.br",
        "quantitativo de serviços por solução tecnológica",
        "formalizado no sei",
        "sgd/me",
        "egd 2020",
        "ações do plano anterior",
    ]
    hits = sum(1 for s in legacy_signals if s in doc_text)

    # Se encontrou 2+ sinais legados E não tem headers padrão de tabela
    standard_signals = ["servico/acao", "produto", "dtpactuada", "anexo de entregas"]
    std_hits = sum(1 for s in standard_signals if s in doc_text)

    if hits >= 2 and std_hits < 2:
        return "legacy_egd2020"
    return None


# --------------- Instância default ----------------------------

converter_accurate = create_converter(accurate=True, ocr=True)

print("Docling configurado (modo ACCURATE + OCR).")
print("Classificadores de tabelas carregados.")
print(f"Produtos no vocabulário: {len(ALL_PRODUTOS)} ({len(CANONICAL_PRODUTOS)} canônicos + {len(LEGACY_PRODUTOS)} eixos legados)")
print(f"Aliases de produto: {len(PRODUTO_ALIASES)}")
print(f"Aliases de eixo: {len(EIXO_ALIASES)}")

## 5. Extração de Tabelas de Risco (Documentos Diretivos)

Extrai e normaliza a tabela de riscos de cada Documento Diretivo. Inclui merge multi-página, recuperação de header-as-data e resolução de ações numéricas.

In [ ]:
# ============================================================
# CÉLULA 7 — Extração de Tabelas de Risco (Documentos Diretivos)
# ============================================================

def _map_risk_columns(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    """Mapeia colunas do DataFrame para nomes canônicos de risco.

    Retorna dict com chaves canônicas e valores = nome real da coluna (ou None).
    """
    canonical = {
        "risco": None,
        "probabilidade": None,
        "impacto": None,
        "tratamento": None,
        "acoes": None,
    }

    # Keywords por campo canônico (prioridade decrescente)
    keyword_map = {
        "risco": ["risco", "evento", "descricao do risco", "descricao", "id do risco"],
        "probabilidade": ["probabilidade", "probabilidade de ocorrer", "prob",
                          "classificacao de probabilidade"],
        "impacto": ["impacto", "severidade", "classificacao de impacto"],
        "tratamento": ["opcao de tratamento", "tratamento", "resposta",
                       "tipo de tratamento", "estrategia"],
        "acoes": ["acoes de tratamento", "descrever acoes", "acoes", "acao",
                  "medidas", "plano de acao"],
    }

    # Limpar headers: tratar quebras de linha e hífens
    def _clean_h(c):
        s = _normalize_header(str(c))
        s = s.replace("- ", "").replace("-\n", "")
        return s
    headers = {str(c): _clean_h(c) for c in df.columns}

    for canon_key, keywords in keyword_map.items():
        best_col = None
        best_score = 0.0
        for col_name, col_norm in headers.items():
            for kw in keywords:
                # Substring match
                if kw in col_norm:
                    score = len(kw) / max(len(col_norm), 1)
                    score = max(score, 0.85)  # substring match gets high score
                    if score > best_score:
                        best_score = score
                        best_col = col_name
            # Fuzzy fallback
            if best_col is None:
                for kw in keywords:
                    ratio = difflib.SequenceMatcher(
                        None, col_norm, strip_accents(kw)
                    ).ratio()
                    if ratio > best_score and ratio >= 0.65:
                        best_score = ratio
                        best_col = col_name

        canonical[canon_key] = best_col

    return canonical


def _is_header_row(row: pd.Series, col_map: Dict[str, Optional[str]]) -> bool:
    """Verifica se uma row é na verdade a linha de cabeçalho repetida."""
    values = []
    for canon_key, col_name in col_map.items():
        if col_name is not None and col_name in row.index:
            values.append(_normalize_header(str(row[col_name])))
    header_keywords = [
        "risco", "probabilidade", "impacto", "tratamento", "acoes",
        "evento", "severidade", "resposta",
    ]
    hits = sum(1 for v in values if any(kw in v for kw in header_keywords))
    return hits >= 2


def _cols_are_data(df: pd.DataFrame) -> bool:
    """Detecta se find_tables/Docling interpretou o 1o risco como header de coluna."""
    if df.shape[1] < 4:
        return False
    col0 = _normalize_header(str(df.columns[0]))
    if len(col0) < 10 or col0.startswith("risco") or col0.startswith("col") or col0 == "nan":
        return False
    col1 = _normalize_header(str(df.columns[1]))
    scale_vals = ["raro", "pouco provavel", "provavel", "muito provavel", "praticamente certo",
                  "baixo", "medio", "alto", "muito alto", "baixa", "media", "alta"]
    return any(sv in strip_accents(col1) for sv in scale_vals)


def _is_risk_data(df: pd.DataFrame) -> bool:
    """Verifica se uma tabela contém valores de escala de risco (tabela de continuação)."""
    if df is None or df.empty or df.shape[1] < 4:
        return False
    all_text = strip_accents(" ".join(str(v).lower() for v in df.values.flatten()))
    scale_vals = ["raro", "pouco provavel", "provavel", "muito provavel", "praticamente certo",
                  "muito baixo", "baixo", "medio", "alto", "muito alto",
                  "mitigar", "eliminar", "transferir", "aceitar", "baixa", "media", "alta"]
    return sum(1 for sv in scale_vals if sv in all_text) >= 2


def _extract_action_list_from_text(doc_text: str) -> dict:
    """Extrai a lista 'Referencial para ações de tratamento do risco' do texto do documento."""
    actions = {}
    for pat in [r"[Rr]eferencial\s+para\s+a[çc][õo]es\s+de\s+tratamento",
                r"[Aa][çc][õo]es\s+de\s+tratamento\s+do\s+risco\s*:"]:
        m = re.search(pat, doc_text)
        if m:
            for line in doc_text[m.end():].split('\n'):
                line = line.strip()
                am = re.match(r"^(\d{1,2})\s*[\.\-\)]\s*(.+)", line)
                if am:
                    actions[am.group(1)] = am.group(2).strip()
                elif actions and not line[0:1].isdigit() and len(actions) > 3:
                    break
            if actions:
                return actions
    return actions


def _resolve_action_refs(acoes_text: str, action_list: dict) -> str:
    """Resolve referências numéricas ('1, 2, 9') para texto completo."""
    if not acoes_text or not action_list:
        return acoes_text
    refs = re.findall(r'\d+', acoes_text)
    if not refs:
        return acoes_text
    tokens = re.split(r'[,;\s]+', acoes_text.strip())
    num_tokens = sum(1 for t in tokens if re.match(r'^\d+$', t.strip()))
    if num_tokens / max(len(tokens), 1) < 0.5:
        return acoes_text  # texto livre, não referência
    resolved = [f"{ref}. {action_list[ref]}" for ref in refs if ref in action_list]
    return " | ".join(resolved) if resolved else acoes_text


def extract_risk_table(
    pdf_path: str, sigla: str
) -> Tuple[List[RiskEntry], List[ProcessingError]]:
    """Extrai a tabela de riscos de um Documento Diretivo.

    Inclui:
    - Merge de tabelas multi-página (header na pág N, dados na pág N+1)
    - Recuperação de riscos que viraram header de coluna em tabelas de continuação
    - Resolução de referências numéricas de ações ('1, 2, 9' → texto completo)
    """
    entries: List[RiskEntry] = []
    errors: List[ProcessingError] = []

    def _try_extract(conv: DocumentConverter) -> Tuple[List[RiskEntry], bool]:
        result = conv.convert(pdf_path)

        # Detectar formato legado
        legacy = detect_legacy_format(result)
        if legacy:
            logger.info(f"[{sigla}] Formato legado detectado ({legacy})")
            errors.append(ProcessingError(
                orgao_sigla=sigla, document_type="diretivo", stage="extraction",
                error_type="legacy_format",
                error_message=f"Formato legado {legacy}: estrutura pode diferir do template atual",
            ))

        # Extrair lista de ações de tratamento do texto do documento
        try:
            doc_text = result.document.export_to_markdown()
        except Exception:
            doc_text = ""
        action_list = _extract_action_list_from_text(doc_text)

        local_entries: List[RiskEntry] = []
        found = False
        risk_ncols = None
        col_map = None

        for table in result.document.tables:
            try:
                df = table.export_to_dataframe()
            except Exception as exc:
                logger.warning(f"[{sigla}] Falha ao exportar tabela: {exc}")
                continue

            if df is None or df.shape[1] < 4:
                continue

            has_header = classify_diretivo_table(df) == "risk_table"
            data_as_header = _cols_are_data(df)
            is_continuation = (risk_ncols and df.shape[1] == risk_ncols
                               and not has_header and _is_risk_data(df))

            if not has_header and not data_as_header and not is_continuation:
                continue

            found = True

            if has_header:
                col_map = _map_risk_columns(df)
                if col_map["risco"] is None and len(df.columns) > 0:
                    col_map["risco"] = str(df.columns[0])
                risk_ncols = len(df.columns)

                # Checar se primeira linha é sub-header
                if len(df) > 0 and _is_header_row(df.iloc[0], col_map):
                    df = df.iloc[1:].reset_index(drop=True)
                if len(df) == 0:
                    continue  # Header-only, dados na próxima tabela

            elif data_as_header:
                # Primeiro risco virou header de coluna — recuperar
                risk_ncols = len(df.columns)
                if col_map is None:
                    # Inferir col_map pela posição
                    col_names = list(df.columns)
                    col_map = {}
                    positions = ["risco", "probabilidade", "impacto", "tratamento", "acoes"]
                    if len(col_names) >= 6:
                        positions = ["id_risco"] + positions
                    for i, field in enumerate(positions):
                        if i < len(col_names):
                            col_map[field] = str(col_names[i])

                # Extrair o risco que virou header
                header_vals = {str(c): normalize_text(str(c)) for c in df.columns}
                risco_raw = normalize_text(str(df.columns[0])) if col_map.get("risco") else ""
                if risco_raw.lower() not in ("nan", "none", "") and not _is_header_row(
                    pd.Series({str(c): str(c) for c in df.columns}), col_map
                ):
                    prob_col = col_map.get("probabilidade", "")
                    imp_col = col_map.get("impacto", "")
                    trat_col = col_map.get("tratamento", "")
                    acoes_col = col_map.get("acoes", "")

                    prob_raw = normalize_text(str(dict(zip([str(c) for c in df.columns], df.columns)).get(prob_col, "")))
                    # Simplificar: usar posição
                    cols_list = [normalize_text(str(c)) for c in df.columns]
                    for var in cols_list:
                        if var.lower() in ("nan", "none"):
                            cols_list[cols_list.index(var)] = ""

                    p_raw = cols_list[1] if len(cols_list) > 1 else ""
                    i_raw = cols_list[2] if len(cols_list) > 2 else ""
                    t_raw = cols_list[3] if len(cols_list) > 3 else ""
                    a_raw = cols_list[4] if len(cols_list) > 4 else ""

                    prob_norm, prob_score = fuzzy_match_scale(p_raw, PROBABILIDADE_SCALE)
                    imp_norm, imp_score = fuzzy_match_scale(i_raw, IMPACTO_SCALE)
                    trat_norm, trat_score = fuzzy_match_scale(t_raw, TRATAMENTO_OPTIONS)
                    acoes_resolved = _resolve_action_refs(a_raw, action_list)

                    local_entries.append(RiskEntry(
                        orgao_sigla=sigla, risco_texto=cols_list[0],
                        probabilidade_original=p_raw,
                        probabilidade_normalizada=prob_norm if prob_score >= 0.70 else "",
                        impacto_original=i_raw,
                        impacto_normalizado=imp_norm if imp_score >= 0.70 else "",
                        tratamento_original=t_raw,
                        tratamento_normalizado=trat_norm if trat_score >= 0.70 else "",
                        acoes_tratamento=acoes_resolved,
                        extraction_confidence="medium",
                        needs_review=True,
                        review_reason="recuperado de header de coluna",
                    ))

            elif is_continuation and col_map:
                # Tabela de continuação — pular sub-headers
                if len(df) > 0 and _is_header_row(df.iloc[0], col_map):
                    df = df.iloc[1:].reset_index(drop=True)

            # Extrair linhas de dados
            if col_map is None:
                continue

            mapped_count = sum(1 for v in col_map.values() if v is not None)

            for idx, row in df.iterrows():
                if _is_header_row(row, col_map):
                    continue

                risco_raw = ""
                if col_map.get("risco") and col_map["risco"] in row.index:
                    risco_raw = normalize_text(str(row[col_map["risco"]]))
                prob_raw = ""
                if col_map.get("probabilidade") and col_map["probabilidade"] in row.index:
                    prob_raw = normalize_text(str(row[col_map["probabilidade"]]))
                imp_raw = ""
                if col_map.get("impacto") and col_map["impacto"] in row.index:
                    imp_raw = normalize_text(str(row[col_map["impacto"]]))
                trat_raw = ""
                if col_map.get("tratamento") and col_map["tratamento"] in row.index:
                    trat_raw = normalize_text(str(row[col_map["tratamento"]]))
                acoes_raw = ""
                if col_map.get("acoes") and col_map["acoes"] in row.index:
                    acoes_raw = normalize_text(str(row[col_map["acoes"]]))

                # Limpar nan
                for var_name in ["risco_raw", "prob_raw", "imp_raw", "trat_raw", "acoes_raw"]:
                    if eval(var_name).lower() in ("nan", "none"):
                        exec(f"{var_name} = ''")

                if not risco_raw and not prob_raw and not imp_raw:
                    continue

                # Normalizar
                prob_norm, prob_score = fuzzy_match_scale(prob_raw, PROBABILIDADE_SCALE)
                imp_norm, imp_score = fuzzy_match_scale(imp_raw, IMPACTO_SCALE)
                trat_norm, trat_score = fuzzy_match_scale(trat_raw, TRATAMENTO_OPTIONS)
                acoes_resolved = _resolve_action_refs(acoes_raw, action_list)

                review_reasons = []
                if prob_raw and prob_score < 0.70:
                    review_reasons.append(f"probabilidade não mapeada: '{prob_raw}'")
                if imp_raw and imp_score < 0.70:
                    review_reasons.append(f"impacto não mapeado: '{imp_raw}'")
                if not risco_raw:
                    review_reasons.append("texto do risco vazio")

                confidence = "high" if mapped_count >= 4 and not review_reasons else \
                             "medium" if mapped_count >= 3 and len(review_reasons) <= 1 else "low"

                local_entries.append(RiskEntry(
                    orgao_sigla=sigla, risco_texto=risco_raw,
                    probabilidade_original=prob_raw,
                    probabilidade_normalizada=prob_norm if prob_score >= 0.70 else "",
                    impacto_original=imp_raw,
                    impacto_normalizado=imp_norm if imp_score >= 0.70 else "",
                    tratamento_original=trat_raw,
                    tratamento_normalizado=trat_norm if trat_score >= 0.70 else "",
                    acoes_tratamento=acoes_resolved,
                    extraction_confidence=confidence,
                    needs_review=len(review_reasons) > 0,
                    review_reason="; ".join(review_reasons) if review_reasons else None,
                ))

        return local_entries, found

    # Tentativa principal: modo ACCURATE
    try:
        entries, found = _try_extract(converter_accurate)
        if not found:
            # Fallback: modo FAST
            logger.info(f"[{sigla}] Tabela de risco não encontrada no modo ACCURATE, tentando FAST...")
            converter_fast = create_converter(accurate=False, ocr=True)
            entries, found = _try_extract(converter_fast)
            if not found:
                errors.append(ProcessingError(
                    orgao_sigla=sigla,
                    document_type="diretivo",
                    stage="extraction",
                    error_type="no_risk_table",
                    error_message=f"Nenhuma tabela de risco encontrada em {os.path.basename(pdf_path)}",
                ))
    except Exception as exc:
        logger.warning(f"[{sigla}] ACCURATE falhou: {exc}. Tentando modo FAST...")
        try:
            converter_fast = create_converter(accurate=False, ocr=True)
            entries, found = _try_extract(converter_fast)
            if not found:
                errors.append(ProcessingError(
                    orgao_sigla=sigla,
                    document_type="diretivo",
                    stage="extraction",
                    error_type="no_risk_table",
                    error_message=f"Nenhuma tabela de risco encontrada (modo FAST) em {os.path.basename(pdf_path)}",
                ))
        except Exception as exc2:
            errors.append(ProcessingError(
                orgao_sigla=sigla,
                document_type="diretivo",
                stage="extraction",
                error_type="docling_failure",
                error_message=f"Docling falhou em ambos os modos: {exc2}",
            ))

    return entries, errors


# --------------- Extração em lote -----------------------------

def extract_all_risks() -> None:
    """Extrai tabelas de risco de todos os órgãos com Documento Diretivo."""
    global all_risks, all_errors

    # Tentar carregar checkpoint (só usa se tiver dados reais)
    cached = load_checkpoint("risks_raw")
    if cached is not None and len(cached[0]) > 0:
        cached_risks, cached_errors, processed_siglas = cached
        all_risks.extend(cached_risks)
        all_errors.extend(cached_errors)
        print(f"  Retomando: {len(cached_risks)} riscos já extraídos de {len(processed_siglas)} órgãos")
    else:
        cached_risks = []
        cached_errors = []
        processed_siglas = set()

    # Órgãos que têm PDF diretivo
    organs_with_pdf = [o for o in all_organs if o.pdf_path_diretivo]
    pending = [o for o in organs_with_pdf if o.sigla not in processed_siglas]

    if not pending:
        print("  Todos os órgãos já processados (checkpoint).")
        return

    print(f"  Processando riscos: {len(pending)} órgãos pendentes "
          f"({len(processed_siglas)} já processados)")

    # Rastreamento de PDFs já processados (para grupos com PDF compartilhado)
    pdf_results_cache: Dict[str, Tuple[List[RiskEntry], List[ProcessingError]]] = {}

    batch_risks: List[RiskEntry] = []
    batch_errors: List[ProcessingError] = []
    count = 0

    for organ in tqdm(pending, desc="Extraindo riscos"):
        sigla = organ.sigla
        pdf_path = organ.pdf_path_diretivo

        if not os.path.isfile(pdf_path):
            err = ProcessingError(
                orgao_sigla=sigla,
                document_type="diretivo",
                stage="extraction",
                error_type="file_not_found",
                error_message=f"PDF não encontrado: {pdf_path}",
            )
            batch_errors.append(err)
            all_errors.append(err)
            processed_siglas.add(sigla)
            count += 1
            continue

        # Verificar se o PDF já foi processado (órgãos agrupados)
        real_path = os.path.realpath(pdf_path)
        if real_path in pdf_results_cache:
            # PDF compartilhado — não duplicar entries (evita dupla contagem).
            # O órgão-cabeça já possui os registros; este membro é apenas
            # marcado como processado.
            owner_sigla = pdf_results_cache[real_path][2]
            processed_siglas.add(sigla)
            logger.info(
                f"[{sigla}] PDF compartilhado com {owner_sigla} — "
                f"riscos atribuídos a {owner_sigla} (sem duplicação)"
            )
        else:
            # Processar PDF
            entries, errs = extract_risk_table(pdf_path, sigla)
            pdf_results_cache[real_path] = (entries, errs, sigla)

            batch_risks.extend(entries)
            all_risks.extend(entries)
            batch_errors.extend(errs)
            all_errors.extend(errs)
            processed_siglas.add(sigla)

            if entries:
                logger.info(f"[{sigla}] {len(entries)} riscos extraídos")
            else:
                logger.info(f"[{sigla}] Nenhum risco extraído")

        count += 1

        # Checkpoint a cada 10 órgãos
        if count % 10 == 0:
            save_checkpoint(
                (cached_risks + batch_risks, cached_errors + batch_errors, processed_siglas),
                "risks_raw",
            )

    # Checkpoint final
    save_checkpoint(
        (cached_risks + batch_risks, cached_errors + batch_errors, processed_siglas),
        "risks_raw",
    )
    print(f"  Extração de riscos concluída.")


# --------------- Execução -------------------------------------
extract_all_risks()

# Resumo
organs_with_risks = set(r.orgao_sigla for r in all_risks)
organs_without_risks = set(
    o.sigla for o in all_organs if o.pdf_path_diretivo
) - organs_with_risks
risk_errors = [e for e in all_errors if e.document_type == "diretivo" and e.stage == "extraction"]

print(f"\n{'='*60}")
print(f"RESUMO — Extração de Riscos")
print(f"{'='*60}")
print(f"  Total de riscos extraídos: {len(all_risks)}")
print(f"  Órgãos com tabela de risco: {len(organs_with_risks)}")
print(f"  Órgãos sem tabela de risco: {len(organs_without_risks)}")
if organs_without_risks:
    print(f"    → {', '.join(sorted(organs_without_risks))}")
print(f"  Erros de extração: {len(risk_errors)}")
needs_review = [r for r in all_risks if r.needs_review]
print(f"  Riscos que precisam revisão: {len(needs_review)}")
confidence_counts = {}
for r in all_risks:
    confidence_counts[r.extraction_confidence] = confidence_counts.get(r.extraction_confidence, 0) + 1
print(f"  Confiança: {confidence_counts}")

## 6. Extração de Tabelas de Entregas (Anexos de Entregas)

Extrai tabelas de entregas pactuadas, concluídas e canceladas de cada Anexo de Entregas.

In [ ]:
# ============================================================
# CÉLULA 8 — Extração de Tabelas de Entregas (Anexos de Entregas)
# ============================================================

def _map_entregas_columns(
    df: pd.DataFrame, tabela_tipo: str
) -> Dict[str, Optional[str]]:
    """Mapeia colunas do DataFrame para nomes canônicos de entregas.

    Args:
        df: DataFrame da tabela extraída.
        tabela_tipo: 'pactuadas', 'concluidas', ou 'canceladas'.

    Returns:
        Dict com chaves canônicas e valores = nome real da coluna (ou None).
    """
    canonical: Dict[str, Optional[str]] = {
        "servico_acao": None,
        "produto": None,
        "eixo": None,
        "area_responsavel": None,
        "data_pactuada": None,
        "data_entrega": None,
        "pactuado": None,
        "justificativa": None,
    }

    keyword_map = {
        "servico_acao": [
            "servico/acao", "servico / acao", "servico", "acao",
            "nome do servico", "servico ou acao",
            # MEC variant: "Serviço /Ação" (espaço antes da barra)
        ],
        "produto": [
            "produto", "produto ptd", "entrega", "produto/entrega",
        ],
        "eixo": [
            "eixo", "eixo ptd", "eixo de transformacao",
        ],
        "area_responsavel": [
            "area responsavel", "arearesponsavel", "area",
            "unidade responsavel", "setor",
            "responsavel",  # colunas partidas: "Area\nResponsavel"
        ],
        "data_pactuada": [
            "dtpactuada", "dt pactuada", "data pactuada", "datapactuada",
            "prazo", "data prevista", "previsao",
            "dtpactuadadtreplanejada",  # ANVISA variant
        ],
        "data_entrega": [
            "dtentrega", "dt entrega", "data entrega", "dataentrega",
            "data de entrega", "data conclusao", "dataconclusao",
            # MD group usa "Data Entrega" como coluna principal
        ],
        "pactuado": [
            "pactuado?", "pactuado ?", "pactuado", "foi pactuado",
        ],
        "justificativa": [
            "justificativa", "motivo", "motivo cancelamento",
            "motivo do cancelamento", "observacao", "obs",
        ],
    }

    # Limpar headers: tratar quebras de linha e hífens de quebra
    def _clean_h(c):
        s = _normalize_header(str(c))
        s = s.replace("- ", "").replace("-\n", "")
        return s
    headers = {str(c): _clean_h(c) for c in df.columns}

    for canon_key, keywords in keyword_map.items():
        best_col = None
        best_score = 0.0

        for col_name, col_norm in headers.items():
            # Remove espaços para comparação compacta
            col_compact = col_norm.replace(" ", "")

            for kw in keywords:
                kw_norm = strip_accents(kw.lower())
                kw_compact = kw_norm.replace(" ", "")

                # Substring match
                if kw_norm in col_norm or kw_compact in col_compact:
                    score = len(kw_norm) / max(len(col_norm), 1)
                    score = max(score, 0.85)
                    if score > best_score:
                        best_score = score
                        best_col = col_name
                    continue

                # Fuzzy match
                ratio = difflib.SequenceMatcher(
                    None, col_norm, kw_norm
                ).ratio()
                if ratio > best_score and ratio >= 0.65:
                    best_score = ratio
                    best_col = col_name

        canonical[canon_key] = best_col

    return canonical


def _clean_cell(value) -> str:
    """Limpa valor de célula: trata NaN, multi-line, espaços extras."""
    if value is None:
        return ""
    s = str(value)
    if s.lower() in ("nan", "none", "nat", ""):
        return ""
    # Multi-line text: juntar com espaço
    s = re.sub(r"[\r\n]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _is_entregas_header_row(row: pd.Series, col_map: Dict[str, Optional[str]]) -> bool:
    """Verifica se a row é uma linha de cabeçalho repetida."""
    values = []
    for canon_key, col_name in col_map.items():
        if col_name is not None and col_name in row.index:
            values.append(_normalize_header(str(row[col_name])))

    header_keywords = [
        "servico", "acao", "produto", "eixo", "area", "responsavel",
        "dtpactuada", "dtentrega", "pactuado", "justificativa",
        "data", "prazo",
    ]
    hits = sum(1 for v in values if any(kw in v for kw in header_keywords))
    return hits >= 2


def extract_entregas_tables(
    pdf_path: str, sigla: str
) -> Tuple[List[DeliveryEntry], List[ProcessingError]]:
    """Extrai tabelas de entregas de um Anexo de Entregas.

    Args:
        pdf_path: Caminho do PDF de entregas.
        sigla: Sigla do órgão.

    Returns:
        (lista de DeliveryEntry, lista de ProcessingError)
    """
    entries: List[DeliveryEntry] = []
    errors: List[ProcessingError] = []

    def _try_extract(conv: DocumentConverter) -> Tuple[List[DeliveryEntry], bool]:
        """Tenta extrair com um conversor. Retorna (entries, found_any_table)."""
        result = conv.convert(pdf_path)

        # Detectar formato legado (EGD 2020-2022)
        legacy = detect_legacy_format(result)
        if legacy:
            logger.info(f"[{sigla}] Formato legado detectado ({legacy}) — tabelas podem diferir do padrão")
            errors.append(ProcessingError(
                orgao_sigla=sigla,
                document_type="entregas",
                stage="extraction",
                error_type="legacy_format",
                error_message=f"Formato legado {legacy}: estrutura pode diferir do template atual",
            ))

        local_entries: List[DeliveryEntry] = []
        found_any = False

        for table in result.document.tables:
            try:
                df = table.export_to_dataframe()
            except Exception as exc:
                logger.warning(f"[{sigla}] Falha ao exportar tabela de entregas: {exc}")
                continue

            if df is None or df.empty:
                continue

            classification = classify_entregas_table(df)

            if classification == "unknown":
                continue

            found_any = True
            col_map = _map_entregas_columns(df, classification)
            mapped_count = sum(1 for v in col_map.values() if v is not None)

            # Mapear tabela_tipo para valor padronizado
            tipo_map = {
                "pactuadas": "pactuada",
                "concluidas": "concluida",
                "canceladas": "cancelada",
            }
            tabela_tipo = tipo_map.get(classification, classification)

            for idx, row in df.iterrows():
                # Pular linhas de cabeçalho repetidas
                if _is_entregas_header_row(row, col_map):
                    continue

                # Extrair valores
                servico_raw = _clean_cell(
                    row.get(col_map["servico_acao"]) if col_map["servico_acao"] else None
                )
                produto_raw = _clean_cell(
                    row.get(col_map["produto"]) if col_map["produto"] else None
                )
                eixo_raw = _clean_cell(
                    row.get(col_map["eixo"]) if col_map["eixo"] else None
                )
                area_raw = _clean_cell(
                    row.get(col_map["area_responsavel"]) if col_map["area_responsavel"] else None
                )
                dt_pact_raw = _clean_cell(
                    row.get(col_map["data_pactuada"]) if col_map["data_pactuada"] else None
                )
                dt_entr_raw = _clean_cell(
                    row.get(col_map["data_entrega"]) if col_map["data_entrega"] else None
                )
                pactuado_raw = _clean_cell(
                    row.get(col_map["pactuado"]) if col_map["pactuado"] else None
                )
                justif_raw = _clean_cell(
                    row.get(col_map["justificativa"]) if col_map["justificativa"] else None
                )

                # Pular linhas completamente vazias
                if not servico_raw and not produto_raw and not eixo_raw:
                    continue

                # Normalizar produto e eixo
                produto_norm = ""
                produto_score = 0.0
                is_outros = produto_raw and produto_raw.strip().lower() in ("outros","outro","outros -","outros –")
                if produto_raw:
                    produto_norm, produto_score = fuzzy_match_produto(produto_raw)
                # Se não matchou canônico mas é "Outros" literal, aceitar
                if produto_score < 0.80 and is_outros:
                    produto_norm = "Outros"
                    produto_score = 1.0

                eixo_norm = ""
                eixo_score = 0.0
                if eixo_raw:
                    eixo_norm, eixo_score = fuzzy_match_eixo(eixo_raw)

                # Se eixo não veio no texto mas o produto foi normalizado, inferir
                if not eixo_norm and produto_norm and produto_score >= 0.80:
                    eixo_norm = PRODUTO_TO_EIXO.get(produto_norm, "")

                # Parsear datas
                dt_pactuada = parse_date(dt_pact_raw) if dt_pact_raw else None
                dt_entrega = parse_date(dt_entr_raw) if dt_entr_raw else None

                # Normalizar pactuado (Sim/Não)
                pactuado_final = None
                if pactuado_raw:
                    p_lower = pactuado_raw.lower().strip()
                    if p_lower in ("sim", "s", "yes", "x"):
                        pactuado_final = "Sim"
                    elif p_lower in ("nao", "não", "n", "no"):
                        pactuado_final = "Não"
                    else:
                        pactuado_final = pactuado_raw

                # Determinar confiança
                review_reasons = []
                if produto_raw and produto_score < 0.70:
                    review_reasons.append(f"produto não mapeado: '{produto_raw}'")
                if eixo_raw and eixo_score < 0.70 and not eixo_norm:
                    review_reasons.append(f"eixo não mapeado: '{eixo_raw}'")
                if not servico_raw and not produto_raw:
                    review_reasons.append("serviço e produto vazios")

                if mapped_count >= 4 and not review_reasons:
                    confidence = "high"
                elif mapped_count >= 3 and len(review_reasons) <= 1:
                    confidence = "medium"
                else:
                    confidence = "low"

                entry = DeliveryEntry(
                    orgao_sigla=sigla,
                    tabela_tipo=tabela_tipo,
                    servico_acao=servico_raw,
                    produto_original=produto_raw,
                    produto_normalizado=produto_norm if produto_score >= 0.70 else "",
                    eixo_original=eixo_raw,
                    eixo_normalizado=eixo_norm,
                    area_responsavel=area_raw if area_raw else None,
                    data_pactuada=dt_pactuada,
                    data_entrega=dt_entrega,
                    pactuado=pactuado_final,
                    justificativa=justif_raw if justif_raw else None,
                    extraction_confidence=confidence,
                    needs_review=len(review_reasons) > 0,
                    review_reason="; ".join(review_reasons) if review_reasons else None,
                )
                local_entries.append(entry)

        return local_entries, found_any

    # Tentativa principal: modo ACCURATE
    try:
        entries, found = _try_extract(converter_accurate)
        if not found:
            logger.info(
                f"[{sigla}] Nenhuma tabela de entregas no modo ACCURATE, tentando FAST..."
            )
            converter_fast = create_converter(accurate=False, ocr=True)
            entries, found = _try_extract(converter_fast)
            if not found:
                errors.append(ProcessingError(
                    orgao_sigla=sigla,
                    document_type="entregas",
                    stage="extraction",
                    error_type="no_entregas_table",
                    error_message=(
                        f"Nenhuma tabela de entregas encontrada em "
                        f"{os.path.basename(pdf_path)}"
                    ),
                ))
    except Exception as exc:
        logger.warning(f"[{sigla}] ACCURATE falhou em entregas: {exc}. Tentando FAST...")
        try:
            converter_fast = create_converter(accurate=False, ocr=True)
            entries, found = _try_extract(converter_fast)
            if not found:
                errors.append(ProcessingError(
                    orgao_sigla=sigla,
                    document_type="entregas",
                    stage="extraction",
                    error_type="no_entregas_table",
                    error_message=(
                        f"Nenhuma tabela de entregas encontrada (modo FAST) em "
                        f"{os.path.basename(pdf_path)}"
                    ),
                ))
        except Exception as exc2:
            errors.append(ProcessingError(
                orgao_sigla=sigla,
                document_type="entregas",
                stage="extraction",
                error_type="docling_failure",
                error_message=f"Docling falhou em ambos os modos: {exc2}",
            ))

    return entries, errors


# --------------- Extração em lote -----------------------------

def extract_all_deliveries() -> None:
    """Extrai tabelas de entregas de todos os órgãos com Anexo de Entregas."""
    global all_deliveries, all_errors

    # Tentar carregar checkpoint (só usa se tiver dados reais)
    cached = load_checkpoint("deliveries_raw")
    if cached is not None and len(cached[0]) > 0:
        cached_deliveries, cached_errors, processed_siglas = cached
        all_deliveries.extend(cached_deliveries)
        all_errors.extend(cached_errors)
        print(
            f"  Retomando: {len(cached_deliveries)} entregas já extraídas "
            f"de {len(processed_siglas)} órgãos"
        )
    else:
        cached_deliveries = []
        cached_errors = []
        processed_siglas = set()

    # Órgãos com PDF de entregas
    organs_with_pdf = [o for o in all_organs if o.pdf_path_entregas]
    pending = [o for o in organs_with_pdf if o.sigla not in processed_siglas]

    if not pending:
        print("  Todos os órgãos já processados (checkpoint).")
        return

    print(
        f"  Processando entregas: {len(pending)} órgãos pendentes "
        f"({len(processed_siglas)} já processados)"
    )

    # Cache de resultados por PDF (para órgãos agrupados com PDF compartilhado)
    pdf_results_cache: Dict[str, Tuple[List[DeliveryEntry], List[ProcessingError]]] = {}

    batch_deliveries: List[DeliveryEntry] = []
    batch_errors: List[ProcessingError] = []
    count = 0

    for organ in tqdm(pending, desc="Extraindo entregas"):
        sigla = organ.sigla
        pdf_path = organ.pdf_path_entregas

        if not os.path.isfile(pdf_path):
            err = ProcessingError(
                orgao_sigla=sigla,
                document_type="entregas",
                stage="extraction",
                error_type="file_not_found",
                error_message=f"PDF não encontrado: {pdf_path}",
            )
            batch_errors.append(err)
            all_errors.append(err)
            processed_siglas.add(sigla)
            count += 1
            continue

        # Verificar se o PDF já foi processado (órgãos agrupados)
        real_path = os.path.realpath(pdf_path)
        if real_path in pdf_results_cache:
            # PDF compartilhado — não duplicar entries (evita dupla contagem).
            # O órgão-cabeça já possui os registros; este membro é apenas
            # marcado como processado.
            owner_sigla = pdf_results_cache[real_path][2]
            processed_siglas.add(sigla)
            logger.info(
                f"[{sigla}] PDF compartilhado com {owner_sigla} — "
                f"entregas atribuídas a {owner_sigla} (sem duplicação)"
            )
        else:
            # Processar PDF
            entries, errs = extract_entregas_tables(pdf_path, sigla)
            pdf_results_cache[real_path] = (entries, errs, sigla)

            batch_deliveries.extend(entries)
            all_deliveries.extend(entries)
            batch_errors.extend(errs)
            all_errors.extend(errs)
            processed_siglas.add(sigla)

            if entries:
                logger.info(f"[{sigla}] {len(entries)} entregas extraídas")
            else:
                logger.info(f"[{sigla}] Nenhuma entrega extraída")

        count += 1

        # Checkpoint a cada 10 órgãos
        if count % 10 == 0:
            save_checkpoint(
                (
                    cached_deliveries + batch_deliveries,
                    cached_errors + batch_errors,
                    processed_siglas,
                ),
                "deliveries_raw",
            )

    # Checkpoint final
    save_checkpoint(
        (
            cached_deliveries + batch_deliveries,
            cached_errors + batch_errors,
            processed_siglas,
        ),
        "deliveries_raw",
    )
    print(f"  Extração de entregas concluída.")


# --------------- Execução -------------------------------------
extract_all_deliveries()

# Resumo
organs_with_deliveries = set(d.orgao_sigla for d in all_deliveries)
organs_without_deliveries = set(
    o.sigla for o in all_organs if o.pdf_path_entregas
) - organs_with_deliveries
delivery_errors = [
    e for e in all_errors
    if e.document_type == "entregas" and e.stage == "extraction"
]

print(f"\n{'='*60}")
print(f"RESUMO — Extração de Entregas")
print(f"{'='*60}")
print(f"  Total de entregas extraídas: {len(all_deliveries)}")
print(f"  Órgãos com entregas: {len(organs_with_deliveries)}")
print(f"  Órgãos sem entregas: {len(organs_without_deliveries)}")
if organs_without_deliveries:
    print(f"    → {', '.join(sorted(organs_without_deliveries))}")
print(f"  Erros de extração: {len(delivery_errors)}")

# Breakdown por tipo de tabela
tipo_counts = {}
for d in all_deliveries:
    tipo_counts[d.tabela_tipo] = tipo_counts.get(d.tabela_tipo, 0) + 1
print(f"  Por tipo: {tipo_counts}")

needs_review = [d for d in all_deliveries if d.needs_review]
print(f"  Entregas que precisam revisão: {len(needs_review)}")

confidence_counts = {}
for d in all_deliveries:
    confidence_counts[d.extraction_confidence] = (
        confidence_counts.get(d.extraction_confidence, 0) + 1
    )
print(f"  Confiança: {confidence_counts}")

## 7. Padronização de Vocabulário

Normaliza produtos, eixos, probabilidades, impactos e tratamentos usando matching em camadas.

In [ ]:
# ============================================================
# CÉLULA 9 — Padronização de Vocabulário
# ============================================================
from typing import List, Tuple, Dict
from collections import Counter


def standardize_deliveries(entries: List[DeliveryEntry]) -> Tuple[List[DeliveryEntry], Dict]:
    """
    Normaliza produto e eixo de cada DeliveryEntry usando matching em camadas.

    Retorna a lista atualizada e um relatório de vocabulário.
    """
    produto_mappings: Dict[str, Dict] = {}   # original -> {normalized, score, count}
    eixo_mappings: Dict[str, Dict] = {}
    unmatched_produtos: List[str] = []
    stats = Counter()  # exact, fuzzy, unmatched

    for entry in entries:
        # --- Produto ---
        prod_orig = normalize_text(entry.produto_original)

        if prod_orig in produto_mappings:
            cached = produto_mappings[prod_orig]
            entry.produto_normalizado = cached["normalized"]
            cached["count"] += 1
            score = cached["score"]
        else:
            matched, score = fuzzy_match_produto(entry.produto_original)
            if score >= 0.85:
                entry.produto_normalizado = matched
            elif score >= 0.70:
                entry.produto_normalizado = matched
                entry.needs_review = True
                entry.review_reason = f"fuzzy match baixo ({score:.2f})"
                entry.extraction_confidence = "medium"
            else:
                entry.produto_normalizado = "UNMATCHED"
                entry.needs_review = True
                entry.review_reason = f"produto não reconhecido (melhor score: {score:.2f})"
                entry.extraction_confidence = "low"

            produto_mappings[prod_orig] = {
                "normalized": entry.produto_normalizado,
                "score": score,
                "count": 1,
            }

        # Track stats
        if score == 1.0:
            stats["exact"] += 1
        elif score >= 0.70:
            stats["fuzzy"] += 1
        else:
            stats["unmatched"] += 1

        # --- Eixo ---
        eixo_orig = normalize_text(entry.eixo_original)

        if eixo_orig in eixo_mappings:
            cached_eixo = eixo_mappings[eixo_orig]
            entry.eixo_normalizado = cached_eixo["normalized"]
            cached_eixo["count"] += 1
        else:
            matched_eixo, eixo_score = fuzzy_match_eixo(entry.eixo_original)
            if eixo_score >= 0.70:
                entry.eixo_normalizado = matched_eixo
            else:
                entry.eixo_normalizado = entry.eixo_original  # keep original
            eixo_mappings[eixo_orig] = {
                "normalized": entry.eixo_normalizado,
                "score": eixo_score,
                "count": 1,
            }

        # --- Cross-validation: produto ↔ eixo ---
        if entry.produto_normalizado != "UNMATCHED" and entry.produto_normalizado in PRODUTO_TO_EIXO:
            canonical_eixo = PRODUTO_TO_EIXO[entry.produto_normalizado]
            if entry.eixo_normalizado != canonical_eixo:
                # Prefer the canonical mapping from the product
                old_eixo = entry.eixo_normalizado
                entry.eixo_normalizado = canonical_eixo
                if not entry.needs_review:
                    entry.needs_review = True
                    entry.review_reason = (
                        f"eixo corrigido por cross-validation: "
                        f"'{old_eixo}' → '{canonical_eixo}'"
                    )
                elif entry.review_reason and "eixo corrigido" not in entry.review_reason:
                    entry.review_reason += (
                        f"; eixo corrigido: '{old_eixo}' → '{canonical_eixo}'"
                    )

    # Build unmatched list
    unmatched_produtos = [
        orig for orig, info in produto_mappings.items()
        if info["normalized"] == "UNMATCHED"
    ]

    vocab_report = {
        "produto_mappings": [
            {
                "original": orig,
                "normalized": info["normalized"],
                "score": info["score"],
                "count": info["count"],
            }
            for orig, info in sorted(produto_mappings.items())
        ],
        "eixo_mappings": [
            {
                "original": orig,
                "normalized": info["normalized"],
                "score": info["score"],
                "count": info["count"],
            }
            for orig, info in sorted(eixo_mappings.items())
        ],
        "unmatched_produtos": unmatched_produtos,
        "match_stats": dict(stats),
    }

    return entries, vocab_report


def standardize_risks(entries: List[RiskEntry]) -> Tuple[List[RiskEntry], Dict]:
    """
    Normaliza probabilidade, impacto e tratamento de cada RiskEntry.

    Retorna a lista atualizada e um relatório de normalização.
    """
    prob_mappings: Dict[str, Dict] = {}
    imp_mappings: Dict[str, Dict] = {}
    trat_mappings: Dict[str, Dict] = {}
    stats = {
        "probabilidade": Counter(),  # exact, fuzzy, unmatched
        "impacto": Counter(),
        "tratamento": Counter(),
    }

    for entry in entries:
        review_reasons = []

        # --- Probabilidade ---
        prob_orig = normalize_text(entry.probabilidade_original)
        if prob_orig in prob_mappings:
            cached = prob_mappings[prob_orig]
            entry.probabilidade_normalizada = cached["normalized"]
            cached["count"] += 1
            p_score = cached["score"]
        else:
            matched, p_score = fuzzy_match_scale(entry.probabilidade_original, PROBABILIDADE_SCALE)
            if p_score >= 0.85:
                entry.probabilidade_normalizada = matched
            elif p_score >= 0.70:
                entry.probabilidade_normalizada = matched
                review_reasons.append(f"probabilidade fuzzy ({p_score:.2f})")
            else:
                entry.probabilidade_normalizada = entry.probabilidade_original
                review_reasons.append(f"probabilidade não reconhecida (score: {p_score:.2f})")
            prob_mappings[prob_orig] = {
                "normalized": entry.probabilidade_normalizada,
                "score": p_score,
                "count": 1,
            }

        if p_score == 1.0:
            stats["probabilidade"]["exact"] += 1
        elif p_score >= 0.70:
            stats["probabilidade"]["fuzzy"] += 1
        else:
            stats["probabilidade"]["unmatched"] += 1

        # --- Impacto ---
        imp_orig = normalize_text(entry.impacto_original)
        if imp_orig in imp_mappings:
            cached = imp_mappings[imp_orig]
            entry.impacto_normalizado = cached["normalized"]
            cached["count"] += 1
            i_score = cached["score"]
        else:
            matched, i_score = fuzzy_match_scale(entry.impacto_original, IMPACTO_SCALE)
            if i_score >= 0.85:
                entry.impacto_normalizado = matched
            elif i_score >= 0.70:
                entry.impacto_normalizado = matched
                review_reasons.append(f"impacto fuzzy ({i_score:.2f})")
            else:
                entry.impacto_normalizado = entry.impacto_original
                review_reasons.append(f"impacto não reconhecido (score: {i_score:.2f})")
            imp_mappings[imp_orig] = {
                "normalized": entry.impacto_normalizado,
                "score": i_score,
                "count": 1,
            }

        if i_score == 1.0:
            stats["impacto"]["exact"] += 1
        elif i_score >= 0.70:
            stats["impacto"]["fuzzy"] += 1
        else:
            stats["impacto"]["unmatched"] += 1

        # --- Tratamento (pode ser múltiplo: separado por ; , /) ---
        trat_orig = normalize_text(entry.tratamento_original)
        if trat_orig in trat_mappings:
            cached = trat_mappings[trat_orig]
            entry.tratamento_normalizado = cached["normalized"]
            cached["count"] += 1
            t_score = cached["score"]
        else:
            # Split by multiple separators
            parts = re.split(r"\s*[;,/]\s*", trat_orig) if trat_orig else []
            normalized_parts = []
            worst_score = 1.0

            for part in parts:
                part = part.strip()
                if not part:
                    continue
                matched, t_sc = fuzzy_match_scale(part, TRATAMENTO_OPTIONS)
                if t_sc >= 0.70:
                    normalized_parts.append(matched)
                else:
                    normalized_parts.append(part)
                    review_reasons.append(f"tratamento '{part}' não reconhecido (score: {t_sc:.2f})")
                worst_score = min(worst_score, t_sc)

            entry.tratamento_normalizado = "; ".join(normalized_parts) if normalized_parts else trat_orig
            t_score = worst_score if parts else 0.0
            trat_mappings[trat_orig] = {
                "normalized": entry.tratamento_normalizado,
                "score": t_score,
                "count": 1,
            }

        if t_score == 1.0:
            stats["tratamento"]["exact"] += 1
        elif t_score >= 0.70:
            stats["tratamento"]["fuzzy"] += 1
        else:
            stats["tratamento"]["unmatched"] += 1

        # --- Set confidence and review flags ---
        scores = [p_score, i_score, t_score]
        min_score = min(scores) if scores else 0.0

        if min_score >= 0.85:
            if not entry.needs_review:
                entry.extraction_confidence = "high"
        elif min_score >= 0.70:
            entry.extraction_confidence = "medium"
            entry.needs_review = True
        else:
            entry.extraction_confidence = "low"
            entry.needs_review = True

        if review_reasons:
            existing = entry.review_reason or ""
            new_reasons = "; ".join(review_reasons)
            entry.review_reason = f"{existing}; {new_reasons}".strip("; ") if existing else new_reasons

    risk_report = {
        "probabilidade_mappings": [
            {"original": orig, "normalized": info["normalized"], "score": info["score"], "count": info["count"]}
            for orig, info in sorted(prob_mappings.items())
        ],
        "impacto_mappings": [
            {"original": orig, "normalized": info["normalized"], "score": info["score"], "count": info["count"]}
            for orig, info in sorted(imp_mappings.items())
        ],
        "tratamento_mappings": [
            {"original": orig, "normalized": info["normalized"], "score": info["score"], "count": info["count"]}
            for orig, info in sorted(trat_mappings.items())
        ],
        "stats": {k: dict(v) for k, v in stats.items()},
    }

    return entries, risk_report


# ---- Execução ----
print("=" * 60)
print("PADRONIZAÇÃO DE VOCABULÁRIO")
print("=" * 60)

# Standardize deliveries
if all_deliveries:
    print(f"\nPadronizando {len(all_deliveries)} entregas...")
    all_deliveries, vocab_report = standardize_deliveries(all_deliveries)

    d_stats = vocab_report["match_stats"]
    d_total = sum(d_stats.values()) or 1
    print(f"\n  Produtos — Match exato: {d_stats.get('exact', 0)} "
          f"({d_stats.get('exact', 0)/d_total*100:.1f}%) | "
          f"Fuzzy: {d_stats.get('fuzzy', 0)} "
          f"({d_stats.get('fuzzy', 0)/d_total*100:.1f}%) | "
          f"Não reconhecidos: {d_stats.get('unmatched', 0)} "
          f"({d_stats.get('unmatched', 0)/d_total*100:.1f}%)")

    print(f"  Termos únicos de produto: {len(vocab_report['produto_mappings'])}")
    print(f"  Termos únicos de eixo:    {len(vocab_report['eixo_mappings'])}")

    if vocab_report["unmatched_produtos"]:
        print(f"\n  Produtos NÃO RECONHECIDOS ({len(vocab_report['unmatched_produtos'])}):")
        for p in vocab_report["unmatched_produtos"][:20]:
            print(f"    - '{p}'")
        if len(vocab_report["unmatched_produtos"]) > 20:
            print(f"    ... e mais {len(vocab_report['unmatched_produtos']) - 20}")
else:
    print("\nNenhuma entrega para padronizar.")
    vocab_report = {
        "produto_mappings": [],
        "eixo_mappings": [],
        "unmatched_produtos": [],
        "match_stats": {"exact": 0, "fuzzy": 0, "unmatched": 0},
    }

# Standardize risks
if all_risks:
    print(f"\nPadronizando {len(all_risks)} riscos...")
    all_risks, risk_report = standardize_risks(all_risks)

    for field_name in ["probabilidade", "impacto", "tratamento"]:
        fstats = risk_report["stats"].get(field_name, {})
        ftotal = sum(fstats.values()) or 1
        print(f"\n  {field_name.capitalize():<16s} — "
              f"Exato: {fstats.get('exact', 0):4d} ({fstats.get('exact', 0)/ftotal*100:.1f}%) | "
              f"Fuzzy: {fstats.get('fuzzy', 0):4d} ({fstats.get('fuzzy', 0)/ftotal*100:.1f}%) | "
              f"Falhou: {fstats.get('unmatched', 0):4d} ({fstats.get('unmatched', 0)/ftotal*100:.1f}%)")

    print(f"\n  Termos únicos — probabilidade: {len(risk_report['probabilidade_mappings'])}, "
          f"impacto: {len(risk_report['impacto_mappings'])}, "
          f"tratamento: {len(risk_report['tratamento_mappings'])}")
else:
    print("\nNenhum risco para padronizar.")
    risk_report = {
        "probabilidade_mappings": [],
        "impacto_mappings": [],
        "tratamento_mappings": [],
        "stats": {},
    }

# Review summary
n_review_del = sum(1 for d in all_deliveries if d.needs_review)
n_review_risk = sum(1 for r in all_risks if r.needs_review)
print(f"\n{'='*60}")
print(f"RESUMO DA PADRONIZAÇÃO")
print(f"  Entregas para revisão: {n_review_del}/{len(all_deliveries)}")
print(f"  Riscos para revisão:   {n_review_risk}/{len(all_risks)}")
print(f"{'='*60}")

# Save checkpoints
save_checkpoint(all_deliveries, "deliveries_standardized")
save_checkpoint(all_risks, "risks_standardized")
save_checkpoint(vocab_report, "vocab_report")
save_checkpoint(risk_report, "risk_report")
print("\nCheckpoints de padronização salvos.")

## 8. Exportação de Dados

Exporta dados processados em CSV e JSON, mais relatório de erros e ajustes.

In [ ]:
# ============================================================
# CÉLULA 10 — Exportação de Dados
# ============================================================
from datetime import datetime
from dataclasses import asdict


def _file_size_str(path: str) -> str:
    """Retorna tamanho do arquivo em formato legível."""
    size = os.path.getsize(path)
    if size < 1024:
        return f"{size} B"
    elif size < 1024 * 1024:
        return f"{size / 1024:.1f} KB"
    else:
        return f"{size / (1024 * 1024):.1f} MB"


def _build_nested_json(entries: list, key_field: str, metadata_extra: dict = None) -> dict:
    """Agrupa entradas por key_field em um JSON com metadata."""
    grouped = {}
    for entry in entries:
        d = asdict(entry)
        key = d.get(key_field, "UNKNOWN")
        if key not in grouped:
            grouped[key] = []
        grouped[key].append(d)

    metadata = {
        "exported_at": datetime.now().isoformat(),
        "total": len(entries),
        "groups": len(grouped),
    }
    if metadata_extra:
        metadata.update(metadata_extra)

    return {"metadata": metadata, "data": grouped}


export_log = []  # (filename, rows, size_str)

# ---- 1. Entregas: CSV e JSON ----
if all_deliveries:
    df_del = pd.DataFrame([asdict(e) for e in all_deliveries])

    csv_path = os.path.join(DIRS["output"], "deliveries.csv")
    df_del.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("deliveries.csv", len(df_del), _file_size_str(csv_path)))

    json_path = os.path.join(DIRS["output"], "deliveries.json")
    nested = _build_nested_json(all_deliveries, "orgao_sigla")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(nested, f, ensure_ascii=False, indent=2)
    export_log.append(("deliveries.json", len(all_deliveries), _file_size_str(json_path)))
else:
    print("Nenhuma entrega para exportar.")

# ---- 2. Riscos: CSV e JSON ----
if all_risks:
    df_risk = pd.DataFrame([asdict(e) for e in all_risks])

    csv_path = os.path.join(DIRS["output"], "risks.csv")
    df_risk.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("risks.csv", len(df_risk), _file_size_str(csv_path)))

    json_path = os.path.join(DIRS["output"], "risks.json")
    nested = _build_nested_json(all_risks, "orgao_sigla")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(nested, f, ensure_ascii=False, indent=2)
    export_log.append(("risks.json", len(all_risks), _file_size_str(json_path)))
else:
    print("Nenhum risco para exportar.")

# ---- 3. Órgãos: CSV ----
if all_organs:
    df_org = pd.DataFrame([asdict(o) for o in all_organs])

    csv_path = os.path.join(DIRS["output"], "organs.csv")
    df_org.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("organs.csv", len(df_org), _file_size_str(csv_path)))
else:
    print("Nenhum órgão para exportar.")

# ---- 4. Relatório de erros: CSV ----
if all_errors:
    df_err = pd.DataFrame([asdict(e) for e in all_errors])

    csv_path = os.path.join(DIRS["output"], "error_report.csv")
    df_err.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("error_report.csv", len(df_err), _file_size_str(csv_path)))
else:
    print("Nenhum erro registrado para exportar.")

# ---- 5. Mapeamento de vocabulário: CSV ----
vocab_rows = []

# Produto mappings
for m in vocab_report.get("produto_mappings", []):
    vocab_rows.append({
        "type": "produto",
        "original": m["original"],
        "normalized": m["normalized"],
        "score": m["score"],
        "count": m["count"],
    })

# Eixo mappings
for m in vocab_report.get("eixo_mappings", []):
    vocab_rows.append({
        "type": "eixo",
        "original": m["original"],
        "normalized": m["normalized"],
        "score": m["score"],
        "count": m["count"],
    })

# Risk field mappings
for field_key in ["probabilidade_mappings", "impacto_mappings", "tratamento_mappings"]:
    field_type = field_key.replace("_mappings", "")
    for m in risk_report.get(field_key, []):
        vocab_rows.append({
            "type": field_type,
            "original": m["original"],
            "normalized": m["normalized"],
            "score": m["score"],
            "count": m["count"],
        })

if vocab_rows:
    df_vocab = pd.DataFrame(vocab_rows)
    csv_path = os.path.join(DIRS["output"], "vocabulary_mapping.csv")
    df_vocab.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("vocabulary_mapping.csv", len(df_vocab), _file_size_str(csv_path)))
else:
    print("Nenhum mapeamento de vocabulário para exportar.")

# ---- 6. Fila de revisão: CSV ----
review_rows = []

for entry in all_deliveries:
    if entry.needs_review:
        review_rows.append({
            "orgao_sigla": entry.orgao_sigla,
            "entry_type": "delivery",
            "field": "produto / eixo",
            "original_value": entry.produto_original,
            "current_value": entry.produto_normalizado,
            "eixo_original": entry.eixo_original,
            "eixo_normalizado": entry.eixo_normalizado,
            "confidence": entry.extraction_confidence,
            "review_reason": entry.review_reason or "",
            "servico_acao": entry.servico_acao,
            "tabela_tipo": entry.tabela_tipo,
        })

for entry in all_risks:
    if entry.needs_review:
        review_rows.append({
            "orgao_sigla": entry.orgao_sigla,
            "entry_type": "risk",
            "field": "probabilidade / impacto / tratamento",
            "original_value": f"P:{entry.probabilidade_original} | I:{entry.impacto_original} | T:{entry.tratamento_original}",
            "current_value": f"P:{entry.probabilidade_normalizada} | I:{entry.impacto_normalizado} | T:{entry.tratamento_normalizado}",
            "eixo_original": "",
            "eixo_normalizado": "",
            "confidence": entry.extraction_confidence,
            "review_reason": entry.review_reason or "",
            "servico_acao": entry.risco_texto[:100] if entry.risco_texto else "",
            "tabela_tipo": "",
        })

if review_rows:
    df_review = pd.DataFrame(review_rows)
    csv_path = os.path.join(DIRS["output"], "review_queue.csv")
    df_review.to_csv(csv_path, index=False, encoding="utf-8-sig")
    export_log.append(("review_queue.csv", len(df_review), _file_size_str(csv_path)))
else:
    print("Nenhum item pendente de revisão.")

# ---- Resumo de exportação ----
print("\n" + "=" * 60)
print("RESUMO DA EXPORTAÇÃO")
print("=" * 60)
print(f"Diretório de saída: {DIRS['output']}\n")
print(f"{'Arquivo':<30s} {'Registros':>10s} {'Tamanho':>10s}")
print("-" * 52)
for fname, rows, size in export_log:
    print(f"{fname:<30s} {rows:>10,d} {size:>10s}")
print("-" * 52)
total_files = len(export_log)
total_rows = sum(r for _, r, _ in export_log)
print(f"{'TOTAL':<30s} {total_rows:>10,d}   ({total_files} arquivos)")
print("=" * 60)

## 9. Análises Estatísticas e Visualizações

Estatísticas descritivas do corpus e visualizações dos dados extraídos.

In [ ]:
# ============================================================
# CÉLULA 11 — Análises Estatísticas e Visualizações
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

fig_dir = os.path.join(DIRS["output"], "figures")
os.makedirs(fig_dir, exist_ok=True)

# Build DataFrames for analysis (reuse if already created in export cell)
if all_deliveries:
    df_del = pd.DataFrame([asdict(e) for e in all_deliveries])
else:
    df_del = pd.DataFrame()

if all_risks:
    df_risk = pd.DataFrame([asdict(e) for e in all_risks])
else:
    df_risk = pd.DataFrame()

if all_organs:
    df_org = pd.DataFrame([asdict(o) for o in all_organs])
else:
    df_org = pd.DataFrame()


# =========================================================
# 1. Coverage Summary (text)
# =========================================================
print("=" * 60)
print("RESUMO DE COBERTURA")
print("=" * 60)

n_total_organs = len(all_organs)
n_with_diretivo = sum(1 for o in all_organs if o.pdf_path_diretivo) if all_organs else 0
n_with_entregas = sum(1 for o in all_organs if o.pdf_path_entregas) if all_organs else 0
n_with_both = sum(1 for o in all_organs if o.pdf_path_diretivo and o.pdf_path_entregas) if all_organs else 0
n_risks = len(all_risks)
n_deliveries = len(all_deliveries)

# Organs that produced data (have at least one risk or delivery)
organs_with_risks = set(r.orgao_sigla for r in all_risks) if all_risks else set()
organs_with_deliveries = set(d.orgao_sigla for d in all_deliveries) if all_deliveries else set()
organs_with_data = organs_with_risks | organs_with_deliveries

print(f"  Total de órgãos:               {n_total_organs}")
print(f"  Com PDF diretivo:              {n_with_diretivo}")
print(f"  Com PDF entregas:              {n_with_entregas}")
print(f"  Com ambos PDFs:                {n_with_both}")
print(f"  Órgãos com riscos extraídos:   {len(organs_with_risks)}")
print(f"  Órgãos com entregas extraídas: {len(organs_with_deliveries)}")
print(f"  Órgãos com algum dado:         {len(organs_with_data)}")
print(f"")
print(f"  Total de riscos extraídos:     {n_risks}")
print(f"  Total de entregas extraídas:   {n_deliveries}")

if n_total_organs > 0:
    risk_rate = len(organs_with_risks) / n_total_organs * 100
    del_rate = len(organs_with_deliveries) / n_total_organs * 100
    print(f"")
    print(f"  Taxa de extração de riscos:    {risk_rate:.1f}% dos órgãos")
    print(f"  Taxa de extração de entregas:  {del_rate:.1f}% dos órgãos")
print(f"  Erros de processamento:        {len(all_errors)}")
print("=" * 60)


# =========================================================
# 2. Bar Chart: Deliveries by Eixo (horizontal, sorted)
# =========================================================
if not df_del.empty and "eixo_normalizado" in df_del.columns:
    eixo_counts = df_del["eixo_normalizado"].value_counts().sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(4, len(eixo_counts) * 0.5)))
    bars = ax.barh(eixo_counts.index, eixo_counts.values, color=sns.color_palette("viridis", len(eixo_counts)))
    ax.set_xlabel("Número de Entregas")
    ax.set_title("Entregas por Eixo Estratégico")
    ax.bar_label(bars, padding=3)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "01_entregas_por_eixo.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Sem dados de entregas para gráfico de eixos.")


# =========================================================
# 3. Heatmap: Probabilidade x Impacto (risk matrix)
# =========================================================
if not df_risk.empty and "probabilidade_normalizada" in df_risk.columns and "impacto_normalizado" in df_risk.columns:
    # Filter to canonical values only for a clean matrix
    df_risk_clean = df_risk[
        df_risk["probabilidade_normalizada"].isin(PROBABILIDADE_SCALE) &
        df_risk["impacto_normalizado"].isin(IMPACTO_SCALE)
    ].copy()

    if not df_risk_clean.empty:
        pivot = df_risk_clean.groupby(
            ["probabilidade_normalizada", "impacto_normalizado"]
        ).size().unstack(fill_value=0)

        # Reindex to canonical order
        pivot = pivot.reindex(index=PROBABILIDADE_SCALE, columns=IMPACTO_SCALE, fill_value=0)

        fig, ax = plt.subplots(figsize=(10, 6))
        sns.heatmap(
            pivot, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.5, ax=ax,
            xticklabels=IMPACTO_SCALE,
            yticklabels=PROBABILIDADE_SCALE,
        )
        ax.set_xlabel("Impacto")
        ax.set_ylabel("Probabilidade")
        ax.set_title("Matriz de Riscos: Probabilidade × Impacto")
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, "04_matriz_riscos.png"), dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Sem riscos com valores canônicos de probabilidade/impacto para heatmap.")
else:
    print("Sem dados de riscos para heatmap.")


# =========================================================
# 4. Top 20 Produtos (horizontal bar chart)
# =========================================================
if not df_del.empty and "produto_normalizado" in df_del.columns:
    prod_counts = df_del["produto_normalizado"].value_counts().head(20).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(5, len(prod_counts) * 0.35)))
    bars = ax.barh(prod_counts.index, prod_counts.values, color=sns.color_palette("mako", len(prod_counts)))
    ax.set_xlabel("Número de Entregas")
    ax.set_title("Top 20 Produtos Mais Frequentes")
    ax.bar_label(bars, padding=3)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "02_top20_produtos.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Sem dados de entregas para gráfico de produtos.")


# =========================================================
# 5. Deliveries by Type (pie chart)
# =========================================================
if not df_del.empty and "tabela_tipo" in df_del.columns:
    tipo_counts = df_del["tabela_tipo"].value_counts()

    if not tipo_counts.empty:
        colors = {"pactuada": "#4CAF50", "concluida": "#2196F3", "cancelada": "#F44336"}
        pie_colors = [colors.get(t, "#9E9E9E") for t in tipo_counts.index]

        fig, ax = plt.subplots(figsize=(8, 8))
        wedges, texts, autotexts = ax.pie(
            tipo_counts.values,
            labels=tipo_counts.index,
            autopct=lambda pct: f"{pct:.1f}%\n({int(pct/100.*tipo_counts.sum())})",
            colors=pie_colors,
            startangle=90,
            textprops={"fontsize": 12},
        )
        ax.set_title("Distribuição de Entregas por Tipo")
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, "05_distribuicao_tipos.png"), dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Sem dados de tipo de tabela para gráfico de pizza.")
else:
    print("Sem dados de entregas para gráfico de tipos.")


# =========================================================
# 6. Deliveries per Organ (horizontal bar, top 30)
# =========================================================
if not df_del.empty and "orgao_sigla" in df_del.columns:
    org_counts = df_del["orgao_sigla"].value_counts().head(30).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(6, len(org_counts) * 0.3)))
    bars = ax.barh(org_counts.index, org_counts.values, color=sns.color_palette("crest", len(org_counts)))
    ax.set_xlabel("Número de Entregas")
    ax.set_title("Top 30 Órgãos por Número de Entregas")
    ax.bar_label(bars, padding=3)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "03_top30_orgaos_entregas.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Sem dados de entregas para gráfico por órgão.")


# =========================================================
# 7. Treatment Options Distribution (bar chart)
# =========================================================
if not df_risk.empty and "tratamento_normalizado" in df_risk.columns:
    # Tratamentos podem ter múltiplos valores separados por ";"
    all_treatments = []
    for val in df_risk["tratamento_normalizado"].dropna():
        parts = [p.strip() for p in str(val).split(";") if p.strip()]
        all_treatments.extend(parts)

    if all_treatments:
        trat_counts = pd.Series(all_treatments).value_counts()

        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(trat_counts.index, trat_counts.values, color=sns.color_palette("Set2", len(trat_counts)))
        ax.set_xlabel("Tipo de Tratamento")
        ax.set_ylabel("Frequência")
        ax.set_title("Distribuição das Opções de Tratamento de Riscos")
        ax.bar_label(bars, padding=3)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, "06_tratamento_riscos.png"), dpi=150, bbox_inches="tight")
        plt.show()
    else:
        print("Sem dados de tratamento para gráfico.")
else:
    print("Sem dados de riscos para gráfico de tratamentos.")


# =========================================================
# 8. Data Quality Dashboard (text)
# =========================================================
print("\n" + "=" * 60)
print("DASHBOARD DE QUALIDADE DOS DADOS")
print("=" * 60)

# --- Missing/empty field rates ---
print("\n--- Campos com maiores taxas de ausência ---")

if not df_del.empty:
    print("\n  ENTREGAS:")
    for col in df_del.columns:
        n_missing = df_del[col].isna().sum() + (df_del[col] == "").sum()
        pct = n_missing / len(df_del) * 100
        if pct > 0:
            print(f"    {col:<30s} {n_missing:>5d} ausentes ({pct:.1f}%)")

if not df_risk.empty:
    print("\n  RISCOS:")
    for col in df_risk.columns:
        n_missing = df_risk[col].isna().sum() + (df_risk[col] == "").sum()
        pct = n_missing / len(df_risk) * 100
        if pct > 0:
            print(f"    {col:<30s} {n_missing:>5d} ausentes ({pct:.1f}%)")

# --- Confidence distribution ---
print("\n--- Distribuição de confiança ---")

if not df_del.empty and "extraction_confidence" in df_del.columns:
    del_conf = df_del["extraction_confidence"].value_counts()
    print("\n  ENTREGAS:")
    for level in ["high", "medium", "low"]:
        count = del_conf.get(level, 0)
        pct = count / len(df_del) * 100
        bar = "█" * int(pct / 2)
        print(f"    {level:<8s} {count:>5d} ({pct:>5.1f}%) {bar}")

if not df_risk.empty and "extraction_confidence" in df_risk.columns:
    risk_conf = df_risk["extraction_confidence"].value_counts()
    print("\n  RISCOS:")
    for level in ["high", "medium", "low"]:
        count = risk_conf.get(level, 0)
        pct = count / len(df_risk) * 100
        bar = "█" * int(pct / 2)
        print(f"    {level:<8s} {count:>5d} ({pct:>5.1f}%) {bar}")

# --- Items needing review ---
n_review_del = df_del["needs_review"].sum() if not df_del.empty and "needs_review" in df_del.columns else 0
n_review_risk = df_risk["needs_review"].sum() if not df_risk.empty and "needs_review" in df_risk.columns else 0

print(f"\n--- Itens pendentes de revisão ---")
print(f"  Entregas: {int(n_review_del)}")
print(f"  Riscos:   {int(n_review_risk)}")
print(f"  Total:    {int(n_review_del + n_review_risk)}")
print("=" * 60)

## Geração de Dados para o Dashboard

Gera os artefatos consumidos pelo dashboard interativo (`index.html`):
- `output/data.js` — 9 constantes JavaScript
- `output/statistics_summary.json`
- `output/coverage_summary.csv`
- `output/pdf_metadata.csv`

In [ ]:
# ============================================================
# CÉLULA 11c — Geração de Dados para o Dashboard
# ============================================================
# Gera data.js (9 constantes JS), statistics_summary.json,
# coverage_summary.csv e pdf_metadata.csv a partir dos dados
# extraídos pelo pipeline.
#
# Depende de: all_organs, all_deliveries, all_risks, all_errors,
#             DIRS, CANONICAL_EIXOS, PROBABILIDADE_SCALE,
#             IMPACTO_SCALE, fuzzy_match_produto
# ============================================================

from collections import Counter, defaultdict
from datetime import datetime
import statistics as stats_mod

# -----------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------

_MONTH_MAP = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04",
    "mai": "05", "jun": "06", "jul": "07", "ago": "08",
    "set": "09", "out": "10", "nov": "11", "dez": "12",
}


def _parse_year_month(date_str: str) -> Optional[str]:
    """Extrai 'YYYY-MM' de formatos variados de data_pactuada.

    Formatos suportados:
      - 'mar. 2025 (v2)'   → '2025-03'
      - 'DD/MM/YYYY'       → 'YYYY-MM'
      - 'MM/YYYY'          → 'YYYY-MM'
    """
    if not date_str or not date_str.strip():
        return None
    s = date_str.strip().lower()

    # Formato 'mes. YYYY ...'
    for abbr, num in _MONTH_MAP.items():
        if s.startswith(abbr):
            m_year = re.search(r"(\d{4})", s)
            if m_year:
                return f"{m_year.group(1)}-{num}"

    # Formato DD/MM/YYYY
    m = re.match(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    if m:
        return f"{m.group(3)}-{m.group(2).zfill(2)}"

    # Formato MM/YYYY
    m = re.match(r"(\d{1,2})/(\d{4})", s)
    if m:
        return f"{m.group(2)}-{m.group(1).zfill(2)}"

    return None


def _read_pdf_metadata(organs: list) -> List[dict]:
    """Lê metadados dos PDFs baixados (tamanho, datas)."""
    rows = []
    for organ in organs:
        for tipo, path in [("diretivo", organ.pdf_path_diretivo),
                           ("entregas", organ.pdf_path_entregas)]:
            if not path or not os.path.exists(path):
                continue
            stat = os.stat(path)
            size_kb = int(stat.st_size / 1024)
            mtime = datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d")

            # Tentar ler data de criação do PDF via pypdf
            creation_date = mtime
            try:
                from pypdf import PdfReader
                reader = PdfReader(path)
                info = reader.metadata
                if info and info.creation_date:
                    creation_date = info.creation_date.strftime("%Y-%m-%d")
            except Exception:
                pass

            rows.append({
                "sigla": organ.sigla,
                "tipo": tipo,
                "data_criacao_pdf": creation_date,
                "data_modificacao_pdf": mtime,
                "vigencia": "",
                "tamanho_kb": size_kb,
            })
    return rows


# -----------------------------------------------------------------
# 1. PTD_STATS  (espelha statistics_summary.json)
# -----------------------------------------------------------------
organs_with_risks = set(r.orgao_sigla for r in all_risks) if all_risks else set()
organs_with_deliveries = set(d.orgao_sigla for d in all_deliveries) if all_deliveries else set()

# Incluir membros de grupo cujo cabeça tem dados (cobertura compartilhada)
for m, head in MEMBER_TO_GROUP.items():
    if head in organs_with_deliveries:
        organs_with_deliveries.add(m)
    if head in organs_with_risks:
        organs_with_risks.add(m)

# Entregas por eixo
eixo_counter = Counter(d.eixo_normalizado for d in all_deliveries if d.eixo_normalizado)
entregas_por_eixo = {e: eixo_counter.get(e, 0) for e in CANONICAL_EIXOS if eixo_counter.get(e, 0) > 0}

# Top 5 produtos
prod_counter = Counter(d.produto_normalizado for d in all_deliveries if d.produto_normalizado)
top5_produtos = dict(prod_counter.most_common(5))

# Distribuições de riscos (apenas valores canônicos)
prob_counter = Counter(r.probabilidade_normalizada for r in all_risks if r.probabilidade_normalizada)
imp_counter = Counter(r.impacto_normalizado for r in all_risks if r.impacto_normalizado)
trat_counter = Counter()
for r in all_risks:
    trat_str = r.tratamento_normalizado or ""
    for t in trat_str.split(";"):
        t = t.strip()
        if t:
            trat_counter[t] += 1

dist_prob = {p: prob_counter.get(p, 0) for p in PROBABILIDADE_SCALE}
dist_imp = {i: imp_counter.get(i, 0) for i in IMPACTO_SCALE}
dist_trat = {t: c for t, c in trat_counter.most_common()}

# Riscos canônicos (com probabilidade e impacto em escalas canônicas)
n_canonicos = sum(1 for r in all_risks
                  if r.probabilidade_normalizada in PROBABILIDADE_SCALE
                  and r.impacto_normalizado in IMPACTO_SCALE)

# Entregas por órgão (para média/mediana)
del_per_organ = Counter(d.orgao_sigla for d in all_deliveries)
del_counts = [c for c in del_per_organ.values()] if del_per_organ else [0]

# PDFs escaneados pendentes
n_scan = sum(1 for e in all_errors if "scan" in e.error_type.lower() or "ocr" in e.error_message.lower())

ptd_stats = {
    "data_execucao": datetime.now().strftime("%Y-%m-%d"),
    "orgaos_total": len(all_organs),
    "entregas_total": len(all_deliveries),
    "entregas_orgaos": len(organs_with_deliveries),
    "riscos_total": len(all_risks),
    "riscos_orgaos": len(organs_with_risks),
    "riscos_canonicos": n_canonicos,
    "pdfs_escaneados_pendentes": n_scan,
    "top5_produtos": top5_produtos,
    "entregas_por_eixo": entregas_por_eixo,
    "distribuicao_probabilidade": dist_prob,
    "distribuicao_impacto": dist_imp,
    "distribuicao_tratamento": dist_trat,
    "media_entregas_por_orgao": round(stats_mod.mean(del_counts), 1) if del_counts else 0,
    "mediana_entregas_por_orgao": int(stats_mod.median(del_counts)) if del_counts else 0,
}

# -----------------------------------------------------------------
# 2. PTD_ORGANS
# -----------------------------------------------------------------
# Pré-computar contagens por órgão
risk_count = Counter(r.orgao_sigla for r in all_risks)
del_count = Counter(d.orgao_sigla for d in all_deliveries)

# Status por órgão
organ_status_map = {}
for e in all_errors:
    key = (e.orgao_sigla, e.document_type)
    if key not in organ_status_map:
        organ_status_map[key] = e.error_type

# Mapa inverso: para membros de grupo sem dados próprios, encontrar o cabeça
# que detém os registros (usando MEMBER_TO_GROUP de 02_config.py)
_head_for = {}
for head_sigla, members in ORGAN_GROUPS.items():
    for m in members:
        if m != head_sigla:
            _head_for[m] = head_sigla

ptd_organs = []
for organ in sorted(all_organs, key=lambda o: o.sigla):
    # Breakdown de eixo e produto
    organ_deliveries = [d for d in all_deliveries if d.orgao_sigla == organ.sigla]
    eixo_bd = dict(Counter(d.eixo_normalizado for d in organ_deliveries if d.eixo_normalizado))
    prod_bd = dict(Counter(d.produto_normalizado for d in organ_deliveries if d.produto_normalizado))

    # Para membros de grupo sem dados próprios, referir ao cabeça
    head = _head_for.get(organ.sigla)
    shares_head = head and del_count.get(organ.sigla, 0) == 0 and del_count.get(head, 0) > 0

    # Status
    if del_count.get(organ.sigla, 0) > 0:
        s_entregas = "ok"
    elif shares_head:
        s_entregas = "compartilhado"
    elif not organ.pdf_path_entregas and not organ.url_entregas:
        s_entregas = "sem_pdf"
    else:
        err_key = (organ.sigla, "entregas")
        s_entregas = organ_status_map.get(err_key, "sem_dados")

    shares_head_r = head and risk_count.get(organ.sigla, 0) == 0 and risk_count.get(head, 0) > 0
    if risk_count.get(organ.sigla, 0) > 0:
        s_riscos = "ok"
    elif shares_head_r:
        s_riscos = "compartilhado"
    elif not organ.pdf_path_diretivo and not organ.url_diretivo:
        s_riscos = "sem_pdf"
    else:
        err_key = (organ.sigla, "diretivo")
        s_riscos = organ_status_map.get(err_key, "sem_dados")

    ptd_organs.append({
        "sigla": organ.sigla,
        "grupo": organ.grupo or "",
        "pdf_diretivo": bool(organ.pdf_path_diretivo or organ.url_diretivo),
        "pdf_entregas": bool(organ.pdf_path_entregas or organ.url_entregas),
        "n_entregas": del_count.get(organ.sigla, 0),
        "n_riscos": risk_count.get(organ.sigla, 0),
        "status_entregas": s_entregas,
        "status_riscos": s_riscos,
        "url_diretivo": organ.url_diretivo or "",
        "url_entregas": organ.url_entregas or "",
        "eixo_breakdown": eixo_bd,
        "produto_breakdown": prod_bd,
    })

# -----------------------------------------------------------------
# 3. PTD_DELIVERIES  (agrupado por sigla)
# -----------------------------------------------------------------
# Cache de scores para evitar recomputar fuzzy match por entrega
_score_cache = {}
def _get_produto_score(original: str, normalizado: str) -> float:
    if not original:
        return 0.0
    if normalizado and normalizado == original:
        return 1.0
    if original not in _score_cache:
        _, s = fuzzy_match_produto(original)
        _score_cache[original] = s
    return _score_cache[original]

ptd_deliveries = defaultdict(list)
for d in all_deliveries:
    pscore = _get_produto_score(d.produto_original or "", d.produto_normalizado or "")

    ptd_deliveries[d.orgao_sigla].append({
        "orgao_sigla": d.orgao_sigla,
        "servico_acao": d.servico_acao or "",
        "produto_original": d.produto_original or "",
        "produto_normalizado": d.produto_normalizado or "",
        "produto_score": round(pscore, 2),
        "eixo_original": d.eixo_original or "",
        "eixo_normalizado": d.eixo_normalizado or "",
        "data_pactuada": d.data_pactuada or "",
        "confidence": d.extraction_confidence or "high",
    })
ptd_deliveries = dict(ptd_deliveries)

# -----------------------------------------------------------------
# 4. PTD_RISKS  (agrupado por sigla, com id_risco sequencial)
# -----------------------------------------------------------------
ptd_risks = defaultdict(list)
for r in all_risks:
    organ_risks = ptd_risks[r.orgao_sigla]
    idx = len(organ_risks)
    id_letter = chr(65 + idx) if idx < 26 else f"R{idx + 1}"

    ptd_risks[r.orgao_sigla].append({
        "orgao_sigla": r.orgao_sigla,
        "id_risco": id_letter,
        "risco_texto": r.risco_texto or "",
        "probabilidade_original": r.probabilidade_original or "",
        "probabilidade_normalizada": r.probabilidade_normalizada or "",
        "impacto_original": r.impacto_original or "",
        "impacto_normalizado": r.impacto_normalizado or "",
        "tratamento_original": r.tratamento_original or "",
        "tratamento_normalizado": r.tratamento_normalizado or "",
        "acoes_original": r.acoes_tratamento or "",
        "acoes_resolvidas": r.acoes_tratamento or "",
    })
ptd_risks = dict(ptd_risks)

# -----------------------------------------------------------------
# 5. PTD_DATES  (sigla → data mais antiga do PDF)
# -----------------------------------------------------------------
pdf_meta_rows = _read_pdf_metadata(all_organs)

ptd_dates = {}
for row in pdf_meta_rows:
    sigla = row["sigla"]
    d = row["data_criacao_pdf"]
    if sigla not in ptd_dates or d < ptd_dates[sigla]:
        ptd_dates[sigla] = d

# Garantir todas as siglas presentes
for organ in all_organs:
    if organ.sigla not in ptd_dates:
        ptd_dates[organ.sigla] = ""

# -----------------------------------------------------------------
# 6-7. PTD_TIMELINE / PTD_TIMELINE_ORGANS
# -----------------------------------------------------------------
month_deliveries = defaultdict(int)
month_organs = defaultdict(set)

for d in all_deliveries:
    ym = _parse_year_month(d.data_pactuada or "")
    if ym:
        month_deliveries[ym] += 1
        month_organs[ym].add(d.orgao_sigla)

ptd_timeline = dict(sorted(month_deliveries.items()))
ptd_timeline_organs = {k: len(v) for k, v in sorted(month_organs.items())}

# -----------------------------------------------------------------
# 8-9. PTD_JACCARD_ORGANS / PTD_JACCARD_MATRIX
# -----------------------------------------------------------------
# Similaridade de Jaccard baseada em conjuntos de produtos por órgão
organ_products = {}
for d in all_deliveries:
    if d.produto_normalizado:
        organ_products.setdefault(d.orgao_sigla, set()).add(d.produto_normalizado)

jaccard_organs = sorted(organ_products.keys())
n = len(jaccard_organs)
jaccard_matrix = [[0.0] * n for _ in range(n)]

for i in range(n):
    si = organ_products[jaccard_organs[i]]
    for j in range(i, n):
        sj = organ_products[jaccard_organs[j]]
        union = len(si | sj)
        if union == 0:
            sim = 0.0
        else:
            sim = round(len(si & sj) / union, 3)
        jaccard_matrix[i][j] = sim
        jaccard_matrix[j][i] = sim

# =================================================================
# Gravar arquivos
# =================================================================

out_dir = DIRS["output"]

# --- data.js ---
js_path = os.path.join(out_dir, "data.js")
with open(js_path, "w", encoding="utf-8") as f:
    f.write(f"const PTD_STATS = {json.dumps(ptd_stats, ensure_ascii=False)};\n")
    f.write(f"const PTD_ORGANS = {json.dumps(ptd_organs, ensure_ascii=False)};\n")
    f.write(f"const PTD_DELIVERIES = {json.dumps(ptd_deliveries, ensure_ascii=False)};\n")
    f.write(f"const PTD_RISKS = {json.dumps(ptd_risks, ensure_ascii=False)};\n")
    f.write(f"const PTD_DATES = {json.dumps(ptd_dates, ensure_ascii=False)};\n")
    f.write(f"const PTD_TIMELINE = {json.dumps(ptd_timeline, ensure_ascii=False)};\n")
    f.write(f"const PTD_TIMELINE_ORGANS = {json.dumps(ptd_timeline_organs, ensure_ascii=False)};\n")
    f.write(f"const PTD_JACCARD_ORGANS = {json.dumps(jaccard_organs, ensure_ascii=False)};\n")
    f.write(f"const PTD_JACCARD_MATRIX = {json.dumps(jaccard_matrix)};\n")
print(f"data.js gravado ({os.path.getsize(js_path) / 1024:.0f} KB)")

# --- statistics_summary.json ---
stats_path = os.path.join(out_dir, "statistics_summary.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(ptd_stats, f, ensure_ascii=False, indent=2)
print(f"statistics_summary.json gravado")

# --- coverage_summary.csv ---
cov_path = os.path.join(out_dir, "coverage_summary.csv")
cov_rows = []
for o in ptd_organs:
    cov_rows.append({
        "sigla": o["sigla"],
        "grupo": o["grupo"],
        "pdf_diretivo": "sim" if o["pdf_diretivo"] else "nao",
        "pdf_entregas": "sim" if o["pdf_entregas"] else "nao",
        "entregas_extraidas": o["n_entregas"],
        "riscos_extraidos": o["n_riscos"],
        "status_entregas": o["status_entregas"],
        "status_riscos": o["status_riscos"],
    })
pd.DataFrame(cov_rows).to_csv(cov_path, index=False, encoding="utf-8-sig")
print(f"coverage_summary.csv gravado ({len(cov_rows)} órgãos)")

# --- pdf_metadata.csv ---
if pdf_meta_rows:
    meta_path = os.path.join(out_dir, "pdf_metadata.csv")
    pd.DataFrame(pdf_meta_rows).to_csv(meta_path, index=False, encoding="utf-8-sig")
    print(f"pdf_metadata.csv gravado ({len(pdf_meta_rows)} PDFs)")
else:
    print("pdf_metadata.csv: nenhum PDF local encontrado (pular)")

print("\nArtefatos do dashboard gerados com sucesso.")

## 10. Suporte a Iterações e Correções Manuais

Ferramentas para revisão manual e aplicação de correções no próximo ciclo.

In [ ]:
# ============================================================
# CÉLULA 12 — Suporte a Iterações e Correções Manuais
# ============================================================
from datetime import datetime
from dataclasses import asdict


def generate_review_queue() -> pd.DataFrame:
    """
    Coleta todos os itens que necessitam revisão manual (riscos, entregas, erros).

    Prioridades (menor número = maior prioridade):
      1 — extraction_failed (erros de processamento)
      2 — low_confidence
      3 — vocabulary_mismatch / unmatched
      4 — medium_confidence / fuzzy match
    """
    rows = []

    # --- Deliveries needing review ---
    for entry in all_deliveries:
        if not entry.needs_review:
            continue

        reason = (entry.review_reason or "").lower()
        if "não reconhecido" in reason or "unmatched" in reason:
            priority = 2 if entry.extraction_confidence == "low" else 3
        elif "fuzzy" in reason:
            priority = 4
        elif "cross-validation" in reason or "corrigido" in reason:
            priority = 3
        else:
            priority = 4

        rows.append({
            "priority": priority,
            "orgao": entry.orgao_sigla,
            "type": "delivery",
            "issue": entry.review_reason or "needs review",
            "original_value": entry.produto_original,
            "current_value": entry.produto_normalizado,
            "confidence": entry.extraction_confidence,
        })

    # --- Risks needing review ---
    for entry in all_risks:
        if not entry.needs_review:
            continue

        reason = (entry.review_reason or "").lower()
        if "não reconhecid" in reason:
            priority = 2 if entry.extraction_confidence == "low" else 3
        elif "fuzzy" in reason:
            priority = 4
        else:
            priority = 4

        rows.append({
            "priority": priority,
            "orgao": entry.orgao_sigla,
            "type": "risk",
            "issue": entry.review_reason or "needs review",
            "original_value": (
                f"P:{entry.probabilidade_original} | "
                f"I:{entry.impacto_original} | "
                f"T:{entry.tratamento_original}"
            ),
            "current_value": (
                f"P:{entry.probabilidade_normalizada} | "
                f"I:{entry.impacto_normalizado} | "
                f"T:{entry.tratamento_normalizado}"
            ),
            "confidence": entry.extraction_confidence,
        })

    # --- Processing errors ---
    for err in all_errors:
        rows.append({
            "priority": 1,
            "orgao": err.orgao_sigla,
            "type": f"error ({err.document_type})",
            "issue": f"[{err.stage}] {err.error_type}: {err.error_message}",
            "original_value": err.url or "",
            "current_value": "",
            "confidence": "failed",
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["priority", "orgao", "type"]).reset_index(drop=True)

    return df


def apply_corrections(corrections_path: str) -> Tuple[int, int]:
    """
    Aplica correções manuais a partir de um CSV.

    Formato do CSV:
      orgao_sigla, entry_type (risk/delivery), field_name, original_value, corrected_value

    Retorna (corrections_applied, corrections_failed).
    """
    if not os.path.exists(corrections_path):
        print(f"Arquivo de correções não encontrado: {corrections_path}")
        return (0, 0)

    df_corr = pd.read_csv(corrections_path, encoding="utf-8-sig")
    required_cols = {"orgao_sigla", "entry_type", "field_name", "original_value", "corrected_value"}
    if not required_cols.issubset(set(df_corr.columns)):
        missing = required_cols - set(df_corr.columns)
        print(f"Colunas faltando no CSV de correções: {missing}")
        return (0, 0)

    applied = 0
    failed = 0

    for _, row in df_corr.iterrows():
        sigla = str(row["orgao_sigla"]).strip()
        entry_type = str(row["entry_type"]).strip().lower()
        field = str(row["field_name"]).strip()
        orig_val = str(row["original_value"]).strip()
        new_val = str(row["corrected_value"]).strip()

        matched = False

        if entry_type == "delivery":
            for entry in all_deliveries:
                if entry.orgao_sigla != sigla:
                    continue
                current = str(getattr(entry, field, None) or "").strip()
                if current == orig_val or (not orig_val and not current):
                    try:
                        setattr(entry, field, new_val)
                        entry.needs_review = False
                        entry.review_reason = f"corrigido manualmente em {datetime.now().isoformat()}"
                        applied += 1
                        matched = True
                    except AttributeError:
                        failed += 1
                    break

        elif entry_type == "risk":
            for entry in all_risks:
                if entry.orgao_sigla != sigla:
                    continue
                current = str(getattr(entry, field, None) or "").strip()
                if current == orig_val or (not orig_val and not current):
                    try:
                        setattr(entry, field, new_val)
                        entry.needs_review = False
                        entry.review_reason = f"corrigido manualmente em {datetime.now().isoformat()}"
                        applied += 1
                        matched = True
                    except AttributeError:
                        failed += 1
                    break

        if not matched:
            failed += 1
            logger.warning(
                f"Correção não aplicada: {sigla}/{entry_type}/{field} "
                f"(valor original '{orig_val}' não encontrado)"
            )

    print(f"Correções aplicadas: {applied}, falhas: {failed}")
    return (applied, failed)


# ---- Gerar fila de revisão ----
print("=" * 60)
print("FILA DE REVISÃO")
print("=" * 60)

review_queue = generate_review_queue()

if not review_queue.empty:
    print(f"\nTotal de itens para revisão: {len(review_queue)}")

    # Breakdown by issue priority
    print("\n--- Por prioridade ---")
    priority_labels = {1: "extraction_failed", 2: "low_confidence", 3: "vocabulary_mismatch", 4: "medium_confidence"}
    for p in sorted(review_queue["priority"].unique()):
        count = (review_queue["priority"] == p).sum()
        label = priority_labels.get(p, f"priority_{p}")
        print(f"  {p}. {label:<25s} {count:>5d}")

    # Breakdown by type
    print("\n--- Por tipo ---")
    for t, count in review_queue["type"].value_counts().items():
        print(f"  {t:<25s} {count:>5d}")

    # Top 10 organs with most review items
    print("\n--- Top 10 órgãos com mais itens para revisão ---")
    top_organs = review_queue["orgao"].value_counts().head(10)
    for org, count in top_organs.items():
        print(f"  {org:<15s} {count:>5d}")

    # Save review queue
    rq_path = os.path.join(DIRS["output"], "review_queue_prioritized.csv")
    review_queue.to_csv(rq_path, index=False, encoding="utf-8-sig")
    print(f"\nFila de revisão salva: {rq_path}")
else:
    print("\nNenhum item pendente de revisão. Todos os dados estão validados.")

# ---- Instructions for corrections ----
print("\n" + "-" * 60)
print("INSTRUÇÕES PARA CORREÇÕES MANUAIS")
print("-" * 60)
print("""
Para aplicar correções no próximo ciclo:

1. Crie um arquivo CSV com as seguintes colunas:
   orgao_sigla, entry_type, field_name, original_value, corrected_value

   Exemplo:
   orgao_sigla,entry_type,field_name,original_value,corrected_value
   MEC,delivery,produto_normalizado,UNMATCHED,Evolução do Serviço
   ANATEL,risk,probabilidade_normalizada,provavel,provável

2. Salve o arquivo em: {output_dir}/corrections.csv

3. Execute: apply_corrections("{output_dir}/corrections.csv")
""".format(output_dir=DIRS["output"]))


# =========================================================
# RESUMO FINAL DO PIPELINE
# =========================================================
print("\n" + "=" * 60)
print("RESUMO FINAL DO PIPELINE")
print("=" * 60)

n_organs = len(all_organs)
n_pdfs = sum(1 for o in all_organs if o.pdf_path_diretivo) + sum(1 for o in all_organs if o.pdf_path_entregas)
n_risks_total = len(all_risks)
n_del_total = len(all_deliveries)
n_errors = len(all_errors)

# Data quality score: % of entries with high confidence
high_conf_del = sum(1 for d in all_deliveries if d.extraction_confidence == "high")
high_conf_risk = sum(1 for r in all_risks if r.extraction_confidence == "high")
total_entries = n_risks_total + n_del_total
quality_score = ((high_conf_del + high_conf_risk) / total_entries * 100) if total_entries > 0 else 0.0

# Items needing review
n_review = len(review_queue) if not review_queue.empty else 0
n_review_organs = review_queue["orgao"].nunique() if not review_queue.empty else 0

# Exported files
output_dir = DIRS["output"]
exported_files = [f for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))] if os.path.isdir(output_dir) else []

print(f"""
  Órgãos processados:           {n_organs}
  PDFs baixados:                {n_pdfs}
  Riscos extraídos:             {n_risks_total}
  Entregas extraídas:           {n_del_total}
  Erros de processamento:       {n_errors}

  Qualidade dos dados:          {quality_score:.1f}% com alta confiança
    - Entregas alta confiança:  {high_conf_del}/{n_del_total}
    - Riscos alta confiança:    {high_conf_risk}/{n_risks_total}

  Arquivos exportados:          {len(exported_files)}
  Diretório de saída:           {output_dir}

  Iteração: {n_review} itens necessitam revisão manual em {n_review_organs} órgãos
""")
print("=" * 60)
print("Pipeline concluído com sucesso.")
print("=" * 60)